In [1]:
import os
import re
import html
import requests
import feedparser
import pandas as pd
import numpy as np
import io
from datetime import datetime

from datetime import datetime
from dotenv import load_dotenv
import mysql.connector

In [2]:
nse_session = requests.Session()

nse_headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/139.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/companies-listing/corporate-filings-announcements"
}

nse_session.headers.update(nse_headers)

# First visit NSE page to establish session/cookies
page_response = nse_session.get(
    "https://www.nseindia.com/companies-listing/corporate-filings-announcements",
    timeout=30
)

print("NSE page status:", page_response.status_code)

NSE page status: 200


In [3]:
from_date = "01-08-2026"
to_date   = "31-08-2026"

api_url = "https://www.nseindia.com/api/corporate-announcements"

params = {
    "index": "equities",
    "from_date": from_date,
    "to_date": to_date
}

response = nse_session.get(
    api_url,
    params=params,
    timeout=30
)

print("Status:", response.status_code)
print("Content length:", len(response.content))
print("Content-Type:", response.headers.get("Content-Type"))

Status: 200
Content length: 14644258
Content-Type: application/json; charset=utf-8


In [4]:
print(response.text[:500])

[{"an_dt":"31-Aug-2026 23:48:18","attFileSize":"344.87 KB","attchmntFile":"https://nsearchives.nseindia.com/corporate/MUTHOOTFIN_31082026234806_AGM_SD.pdf","attchmntText":"Muthoot Finance Limited has informed the Exchange regarding Proceedings of Annual General Meeting held on August 31, 2026","bflag":null,"csvName":null,"desc":"Shareholders meeting","difference":"00:00:01","dt":"31082026234818","exchdisstime":"31-Aug-2026 23:48:19","fileSize":"344.87 KB","hasXbrl":true,"old_new":null,"orgid":nu


In [5]:
data = response.json()

print(type(data))

if isinstance(data, list):
    print("Records:", len(data))
    display(pd.DataFrame(data).head())

elif isinstance(data, dict):
    print("Keys:", data.keys())

<class 'list'>
Records: 21024


,an_dt,attFileSize,attchmntFile,attchmntText,bflag,csvName,desc,difference,dt,exchdisstime,fileSize,hasXbrl,old_new,orgid,seq_id,smIndustry,sm_isin,sm_name,sort_date,symbol
0,31-Aug-2026 23:48:18,344.87 KB,https://nsearchives.nseindia.com/corporate/MUT...,Muthoot Finance Limited has informed the Excha...,None,None,Shareholders meeting,00:00:01,31082026234818,31-Aug-2026 23:48:19,344.87 KB,True,None,None,106763396,Finance,INE414G01012,Muthoot Finance Limited,2026-08-31 23:48:18,MUTHOOTFIN
1,31-Aug-2026 23:36:42,557.77 KB,https://nsearchives.nseindia.com/corporate/GIC...,General Insurance Corporation of India has inf...,None,None,Shareholders meeting,00:00:01,31082026233642,31-Aug-2026 23:36:43,557.77 KB,True,None,None,106763385,None,INE481Y01014,General Insurance Corporation of India,2026-08-31 23:36:42,GICRE
2,31-Aug-2026 23:36:14,2.01 MB,https://nsearchives.nseindia.com/corporate/Dim...,Just Dial Limited has informed the Exchange re...,None,None,Change in Director(s),00:00:01,31082026233614,31-Aug-2026 23:36:15,2.01 MB,True,None,None,106763384,None,INE599M01018,Just Dial Limited,2026-08-31 23:36:14,JUSTDIAL
3,31-Aug-2026 23:31:38,2.01 MB,https://nsearchives.nseindia.com/corporate/chi...,Just Dial Limited has submitted the Exchange a...,None,None,Shareholders meeting,00:00:01,31082026233138,31-Aug-2026 23:31:39,2.01 MB,True,None,None,106763382,None,INE599M01018,Just Dial Limited,2026-08-31 23:31:38,JUSTDIAL
4,31-Aug-2026 23:27:56,298.88 KB,https://nsearchives.nseindia.com/corporate/E2E...,E2E Networks Limited has informed the Exchange...,None,None,Press Release,00:00:01,31082026232756,31-Aug-2026 23:27:57,298.88 KB,True,None,None,106763381,None,INE255Z01019,E2E Networks Limited,2026-08-31 23:27:56,E2E


In [6]:
# Convert NSE response to DataFrame
nse_df = pd.DataFrame(data)

print("Rows:", len(nse_df))
print("\nColumns:")
for col in nse_df.columns:
    print("-", col)

print("\nMissing values:")
display(nse_df.isna().sum().sort_values(ascending=False))

print("\nUnique companies:")
print("Symbols:", nse_df["symbol"].nunique())
print("ISINs:", nse_df["sm_isin"].nunique())
print("Company names:", nse_df["sm_name"].nunique())

print("\nDate range:")
print("Oldest:", nse_df["an_dt"].min())
print("Latest:", nse_df["an_dt"].max())

print("\nAnnouncement types:")

Rows: 21024

Columns:
- an_dt
- attFileSize
- attchmntFile
- attchmntText
- bflag
- csvName
- desc
- difference
- dt
- exchdisstime
- fileSize
- hasXbrl
- old_new
- orgid
- seq_id
- smIndustry
- sm_isin
- sm_name
- sort_date
- symbol

Missing values:


bflag           21024
csvName         21024
orgid           21024
old_new         21024
smIndustry      12981
an_dt               0
hasXbrl             0
sort_date           0
sm_name             0
sm_isin             0
seq_id              0
fileSize            0
attFileSize         0
exchdisstime        0
dt                  0
difference          0
desc                0
attchmntText        0
attchmntFile        0
symbol              0
dtype: int64


Unique companies:
Symbols: 2329
ISINs: 2328
Company names: 2326

Date range:
Oldest: 01-Aug-2026 08:11:24
Latest: 31-Aug-2026 23:48:18

Announcement types:


In [7]:
print("\nDate range:")
print("Oldest:", nse_df["an_dt"].min())
print("Latest:", nse_df["an_dt"].max())

print("\nAnnouncement types:")
display(
    nse_df["desc"]
    .value_counts()
    .head(30)
)


Date range:
Oldest: 01-Aug-2026 08:11:24
Latest: 31-Aug-2026 23:48:18

Announcement types:


desc
Analysts/Institutional Investor Meet/Con. Call Updates                       3442
Copy of Newspaper Publication                                                3083
General Updates                                                              2210
Outcome of Board Meeting                                                     2188
Shareholders meeting                                                         2015
Updates                                                                      1418
Press Release                                                                 966
Investor Presentation                                                         724
Record Date                                                                   502
Appointment                                                                   502
Change in Management                                                          346
Monitoring Agency Report                                                      274
Resignation

In [11]:
news_df = nse_df.copy()

news_df = news_df.rename(columns={
    "an_dt": "published_at",
    "sm_isin": "isin",
    "sm_name": "company_name",
    "desc": "event_type",
    "attchmntFile": "attachment_url",
    "attchmntText": "attachment_text",
    "seq_id": "nse_seq_id",
    "smIndustry": "industry",
    "sort_date": "sort_date",
})

news_df = news_df[
    [
        "nse_seq_id",
        "symbol",
        "isin",
        "company_name",
        "event_type",
        "published_at",
        "sort_date",
        "attachment_url",
        "attachment_text",
        "industry",
        "hasXbrl"
    ]
]

print("Rows:", len(news_df))
display(news_df.head())

Rows: 21024


,nse_seq_id,symbol,isin,company_name,event_type,published_at,sort_date,attachment_url,attachment_text,industry,hasXbrl
0,106763396,MUTHOOTFIN,INE414G01012,Muthoot Finance Limited,Shareholders meeting,31-Aug-2026 23:48:18,2026-08-31 23:48:18,https://nsearchives.nseindia.com/corporate/MUT...,Muthoot Finance Limited has informed the Excha...,Finance,True
1,106763385,GICRE,INE481Y01014,General Insurance Corporation of India,Shareholders meeting,31-Aug-2026 23:36:42,2026-08-31 23:36:42,https://nsearchives.nseindia.com/corporate/GIC...,General Insurance Corporation of India has inf...,None,True
2,106763384,JUSTDIAL,INE599M01018,Just Dial Limited,Change in Director(s),31-Aug-2026 23:36:14,2026-08-31 23:36:14,https://nsearchives.nseindia.com/corporate/Dim...,Just Dial Limited has informed the Exchange re...,None,True
3,106763382,JUSTDIAL,INE599M01018,Just Dial Limited,Shareholders meeting,31-Aug-2026 23:31:38,2026-08-31 23:31:38,https://nsearchives.nseindia.com/corporate/chi...,Just Dial Limited has submitted the Exchange a...,None,True
4,106763381,E2E,INE255Z01019,E2E Networks Limited,Press Release,31-Aug-2026 23:27:56,2026-08-31 23:27:56,https://nsearchives.nseindia.com/corporate/E2E...,E2E Networks Limited has informed the Exchange...,None,True


In [12]:
import mysql.connector
import os
from dotenv import load_dotenv

load_dotenv("/Users/amit/Desktop/InvestIQ/backend/.env")

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor(dictionary=True)

cursor.execute("""
    SELECT
        id,
        symbol,
        name,
        isin
    FROM companies
    WHERE isin IS NOT NULL
""")

companies_df = pd.DataFrame(cursor.fetchall())

cursor.close()
conn.close()

print("Companies in DB:", len(companies_df))
display(companies_df.head())

Companies in DB: 3117


,id,symbol,name,isin
0,5,20MICRONS,20 Microns Limited,INE144J01027
1,6,21STCENMGM,21st Century Management Services Limited,INE253B01015
2,7,360ONE,360 ONE WAM LIMITED,INE466L01038
3,8,3BBLACKBIO,3B Blackbio Dx Limited,INE994E01018
4,9,3IINFOLTD,3i Infotech Limited,INE748C01038


In [13]:
news_df = news_df.merge(
    companies_df[["id", "symbol", "isin"]],
    on="isin",
    how="left",
    suffixes=("", "_db")
)

news_df = news_df.rename(columns={
    "id": "company_id"
})

print("Total NSE announcements:", len(news_df))

print(
    "Mapped to InvestIQ:",
    news_df["company_id"].notna().sum()
)

print(
    "Unmapped:",
    news_df["company_id"].isna().sum()
)

print(
    "Mapping percentage:",
    round(
        news_df["company_id"].notna().mean() * 100,
        2
    ),
    "%"
)

Total NSE announcements: 21024
Mapped to InvestIQ: 16448
Unmapped: 4576
Mapping percentage: 78.23 %


In [14]:
unmapped = (
    news_df[news_df["company_id"].isna()]
    [["symbol", "isin", "company_name"]]
    .drop_duplicates()
)

print("Unique unmapped companies:", len(unmapped))

display(unmapped.head(50))

Unique unmapped companies: 523


,symbol,isin,company_name
4,E2E,INE255Z01019,E2E Networks Limited
11,BRITANNIA,INE216A01014,Britannia Industries Limited
14,PERSISTENT,INE262H01013,Persistent Systems Limited
16,LICHSGFIN,INE115A01018,LIC Housing Finance Limited
20,TATASTEEL,INE081A01012,Tata Steel Limited
21,NAZARA,INE418L01021,Nazara Technologies Limited
26,AXISBANK,INE238A01026,Axis Bank Limited
35,PUNJLLOYD,INE701B01021,Punj Lloyd Limited
37,FILATEX,INE816B01019,Filatex India Limited
45,BANKBARODA,INE028A01013,Bank of Baroda


In [15]:
# Check whether the supposedly unmapped ISINs
# actually exist in our companies table

unmapped = news_df[news_df["company_id"].isna()].copy()

# Normalize ISINs
unmapped["isin_clean"] = (
    unmapped["isin"]
    .astype(str)
    .str.strip()
    .str.upper()
)

companies_df["isin_clean"] = (
    companies_df["isin"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Check direct ISIN match
isin_exists = unmapped["isin_clean"].isin(
    companies_df["isin_clean"]
)

print("Unmapped rows:", len(unmapped))
print("Unmapped rows whose ISIN exists in DB:", isin_exists.sum())
print(
    "Percentage:",
    round(isin_exists.mean() * 100, 2),
    "%"
)

Unmapped rows: 4576
Unmapped rows whose ISIN exists in DB: 0
Percentage: 0.0 %


In [16]:
debug_match = unmapped[
    isin_exists
][
    ["symbol", "isin", "company_name"]
].drop_duplicates()

print("Companies that SHOULD have matched:", len(debug_match))

display(debug_match.head(50))

Companies that SHOULD have matched: 0


,symbol,isin,company_name


In [19]:
# Inspect unmapped NSE announcements directly from the original NSE dataframe

unmapped_symbols = (
    news_df[news_df["company_id"].isna()]
    ["symbol"]
    .drop_duplicates()
)

print("Unique unmapped symbols:", len(unmapped_symbols))

display(
    nse_df[
        nse_df["symbol"].isin(unmapped_symbols)
    ][
        ["symbol", "sm_isin", "sm_name", "smIndustry"]
    ]
    .drop_duplicates()
    .head(100)
)

Unique unmapped symbols: 523


,symbol,sm_isin,sm_name,smIndustry
4,E2E,INE255Z01019,E2E Networks Limited,None
11,BRITANNIA,INE216A01014,Britannia Industries Limited,Food And Food Processing
14,PERSISTENT,INE262H01013,Persistent Systems Limited,Computers - Software
16,LICHSGFIN,INE115A01018,LIC Housing Finance Limited,Finance - Housing
20,TATASTEEL,INE081A01012,Tata Steel Limited,Steel And Steel Products
...,...,...,...,...
601,JTLIND,INE391J01024,JTL INDUSTRIES LIMITED,None
615,TALBROAUTO,INE187D01011,Talbros Automotive Components Limited,Auto Ancillaries
616,BALAXI,INE618N01014,BALAXI PHARMACEUTICALS LIMITED,None
618,GPIL,INE177H01013,Godawari Power And Ispat limited,Steel And Steel Products


In [20]:
# Compare NSE symbols against InvestIQ symbols

nse_symbols = set(
    nse_df["symbol"]
    .astype(str)
    .str.strip()
    .str.upper()
)

db_symbols = set(
    companies_df["symbol"]
    .astype(str)
    .str.strip()
    .str.upper()
)

common_symbols = nse_symbols & db_symbols
only_nse_symbols = nse_symbols - db_symbols

print("NSE unique symbols:", len(nse_symbols))
print("DB unique symbols:", len(db_symbols))
print("Common symbols:", len(common_symbols))
print("Only NSE symbols:", len(only_nse_symbols))

print("\nOnly NSE examples:")
print(list(sorted(only_nse_symbols))[:100])

NSE unique symbols: 2329
DB unique symbols: 3117
Common symbols: 2264
Only NSE symbols: 65

Only NSE examples:
['ABHISHEK', 'AIFL', 'AJRINFRA', 'ALPSINDUS', 'ARSSINFRA', 'ASIL', 'ATCOM', 'ATNINTER', 'AUGMONT', 'BALLARPUR', 'BGLOBAL', 'BHARATIDIL', 'BILENERGY', 'BKMINDST', 'BLUEBLENDS', 'CANDC', 'CMICABLES', 'CRESTO', 'DHARSUGAR', 'DIL', 'ERAINFRA', 'EROSMEDIA', 'FCONSUMER', 'GAJA', 'GAMMONIND', 'GANGOTRI', 'GBGLOBAL', 'GFSTEELS', 'GOENKA', 'HORIZONIND', 'ICSA', 'INDLMETER', 'INDUSFILA', 'JBFIND', 'KDGREEN', 'KSERASERA', 'KSOILS', 'KUNDANMM', 'LALITHAA', 'LEEL', 'MBECL', 'MELSTAR', 'NAGAFERT', 'NITINFIRE', 'OMKARCHEM', 'PODDARHOUS', 'PUNJLLOYD', 'QUINTEGRA', 'RMCL', 'RUSHABEAR', 'SABEVENTS', 'SECURCRED', 'SHANKESH', 'SKIL', 'SRPL', 'SUNSHINE', 'TALWALKARS', 'TECHIN', 'TEMPSENS', 'UNIVAFOODS', 'VALECHAENG', 'VALUEIND', 'VIDEOIND', 'WINSOME', 'XLENERGY']


In [21]:
# Create a clean list of the 523 unmapped symbols

unmapped_symbols_df = (
    news_df[news_df["company_id"].isna()]
    [
        [
            "symbol",
            "isin",
            "company_name"
        ]
    ]
    .drop_duplicates()
    .sort_values("symbol")
)

print("Unique unmapped symbols:", len(unmapped_symbols_df))

display(unmapped_symbols_df.head(100))

Unique unmapped symbols: 523


,symbol,isin,company_name
8630,20MICRONS,INE144J01019,20 Microns Limited
1184,360ONE,INE466L01020,360 ONE WAM LIMITED
401,AAKASH,INE087Z01016,Aakash Exploration Services Limited
7153,AARTECH,INE01C001018,Aartech Solonics Limited
7149,ABHISHEK,INE004I01017,Abhishek Corporation Limited
...,...,...,...
8939,BTML,INE0EEJ01015,Bodhi Tree Multimedia Limited
980,BURNPUR,INE817H01014,Burnpur Cement Limited
534,CAMS,INE596I01012,Computer Age Management Services Limited
819,CANBK,INE476A01014,Canara Bank


In [22]:
from difflib import get_close_matches

db_symbol_list = companies_df["symbol"].dropna().astype(str).str.upper().tolist()

matches = []

for symbol in unmapped_symbols_df["symbol"].astype(str).str.upper():
    close = get_close_matches(
        symbol,
        db_symbol_list,
        n=3,
        cutoff=0.75
    )

    matches.append({
        "nse_symbol": symbol,
        "possible_db_symbols": ", ".join(close)
    })

symbol_matches_df = pd.DataFrame(matches)

display(symbol_matches_df.head(100))


,nse_symbol,possible_db_symbols
0,20MICRONS,20MICRONS
1,360ONE,360ONE
2,AAKASH,"AAKASH, AKASH, SAAKSHI"
3,AARTECH,"AARTECH, AAATECH, TATATECH"
4,ABHISHEK,
...,...,...
95,BTML,"BTML, TTML, ETML"
96,BURNPUR,BURNPUR
97,CAMS,"CAMS, CAMPUS, ASMS"
98,CANBK,CANBK


In [23]:
db_names = companies_df[
    ["id", "symbol", "name", "isin"]
].dropna(subset=["name"]).copy()

db_name_list = db_names["name"].astype(str).tolist()

name_matches = []

for _, row in unmapped_symbols_df.iterrows():

    close = get_close_matches(
        str(row["company_name"]),
        db_name_list,
        n=3,
        cutoff=0.75
    )

    name_matches.append({
        "nse_symbol": row["symbol"],
        "nse_name": row["company_name"],
        "possible_db_names": " | ".join(close)
    })

name_matches_df = pd.DataFrame(name_matches)

display(name_matches_df.head(100))

,nse_symbol,nse_name,possible_db_names
0,20MICRONS,20 Microns Limited,20 Microns Limited | Marsons Limited | Nirlon ...
1,360ONE,360 ONE WAM LIMITED,360 ONE WAM LIMITED | SONAM LIMITED
2,AAKASH,Aakash Exploration Services Limited,Aakash Exploration Services Limited
3,AARTECH,Aartech Solonics Limited,Aartech Solonics Limited
4,ABHISHEK,Abhishek Corporation Limited,Hitech Corporation Limited | Abhishek Integrat...
...,...,...,...
95,BTML,Bodhi Tree Multimedia Limited,Bodhi Tree Multimedia Limited
96,BURNPUR,Burnpur Cement Limited,Burnpur Cement Limited | Orient Cement Limited...
97,CAMS,Computer Age Management Services Limited,Computer Age Management Services Limited | Rap...
98,CANBK,Canara Bank,Canara Bank


In [24]:
# Check whether each DB symbol identifies only one company

symbol_counts = (
    companies_df
    .groupby("symbol")
    .size()
    .sort_values(ascending=False)
)

print("Total DB symbols:", len(symbol_counts))
print("Symbols mapping to >1 company:", (symbol_counts > 1).sum())

display(
    symbol_counts[symbol_counts > 1].head(30)
)

Total DB symbols: 3117
Symbols mapping to >1 company: 0


Series([], dtype: int64)

In [25]:
# Create clean DB symbol lookup

company_symbol_lookup = (
    companies_df[
        ["id", "symbol", "name", "isin"]
    ]
    .dropna(subset=["symbol"])
    .copy()
)

company_symbol_lookup["symbol"] = (
    company_symbol_lookup["symbol"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Only use symbols that uniquely identify one company
unique_symbol_lookup = (
    company_symbol_lookup
    .groupby("symbol")
    .filter(lambda x: len(x) == 1)
)

print(
    "Unique DB symbols available for mapping:",
    unique_symbol_lookup["symbol"].nunique()
)

Unique DB symbols available for mapping: 3117


In [26]:
# Normalize NSE symbol

news_df["symbol_clean"] = (
    news_df["symbol"]
    .astype(str)
    .str.strip()
    .str.upper()
)

symbol_map = (
    unique_symbol_lookup
    .set_index("symbol")["id"]
    .to_dict()
)

news_df["company_id_symbol"] = (
    news_df["symbol_clean"]
    .map(symbol_map)
)

print("Total announcements:", len(news_df))

print(
    "Mapped by symbol:",
    news_df["company_id_symbol"].notna().sum()
)

print(
    "Still unmapped:",
    news_df["company_id_symbol"].isna().sum()
)

print(
    "Mapping percentage:",
    round(
        news_df["company_id_symbol"].notna().mean() * 100,
        2
    ),
    "%"
)

Total announcements: 21024
Mapped by symbol: 20773
Still unmapped: 251
Mapping percentage: 98.81 %


In [27]:
print("Original ISIN mapping:")
print(
    news_df["company_id"].notna().sum()
)

print("\nSymbol mapping:")
print(
    news_df["company_id_symbol"].notna().sum()
)

Original ISIN mapping:
16448

Symbol mapping:
20773


In [28]:
symbol_only = news_df[
    news_df["company_id"].isna()
    &
    news_df["company_id_symbol"].notna()
].copy()

print(
    "Recovered using symbol:",
    len(symbol_only)
)

display(
    symbol_only[
        [
            "symbol",
            "isin",
            "company_name",
            "company_id_symbol"
        ]
    ]
    .drop_duplicates()
    .head(50)
)

Recovered using symbol: 4327


,symbol,isin,company_name,company_id_symbol
4,E2E,INE255Z01019,E2E Networks Limited,640.0
11,BRITANNIA,INE216A01014,Britannia Industries Limited,401.0
14,PERSISTENT,INE262H01013,Persistent Systems Limited,1691.0
16,LICHSGFIN,INE115A01018,LIC Housing Finance Limited,1308.0
20,TATASTEEL,INE081A01012,Tata Steel Limited,2251.0
21,NAZARA,INE418L01021,Nazara Technologies Limited,1526.0
26,AXISBANK,INE238A01026,Axis Bank Limited,263.0
37,FILATEX,INE816B01019,Filatex India Limited,734.0
45,BANKBARODA,INE028A01013,Bank of Baroda,302.0
47,DIXON,INE935N01012,Dixon Technologies (India) Limited,609.0


In [29]:
news_df["final_company_id"] = (
    news_df["company_id"]
    .fillna(news_df["company_id_symbol"])
)

In [30]:
still_unmapped = news_df[
    news_df["company_id_symbol"].isna()
].copy()

print("Unmapped announcements:", len(still_unmapped))
print(
    "Unique unmapped symbols:",
    still_unmapped["symbol"].nunique()
)

display(
    still_unmapped[
        [
            "symbol",
            "isin",
            "company_name"
        ]
    ]
    .drop_duplicates()
    .sort_values("symbol")
)

Unmapped announcements: 251
Unique unmapped symbols: 65


,symbol,isin,company_name
7149,ABHISHEK,INE004I01017,Abhishek Corporation Limited
7042,AIFL,INE428O01016,Ashapura Intimates Fashion Limited
726,AJRINFRA,INE181G01025,AJR INFRA AND TOLLING LIMITED
2857,ALPSINDUS,INE093B01015,Alps Industries Limited
497,ARSSINFRA,INE267I01010,ARSS Infrastructure Projects Limited
...,...,...,...
1044,VALECHAENG,INE624C01015,Valecha Engineering Limited
6367,VALUEIND,INE352A01017,Value Industries Limited
6365,VIDEOIND,INE703A01011,Videocon Industries Limited
451,WINSOME,INE784B01035,Winsome Yarns Limited


In [31]:
print("Unmapped announcement types:")

display(
    still_unmapped["event_type"]
    .value_counts()
    .head(30)
)

Unmapped announcement types:


event_type
Outcome of Board Meeting                                   60
Copy of Newspaper Publication                              40
General Updates                                            25
Updates                                                    24
Corporate Insolvency Resolution Process                    14
Shareholders meeting                                       13
Change in Management                                       10
Trading Window                                              7
Appointment                                                 5
Resignation of Director/KMP/SMP                             5
Integrated Filing- Financial                                5
Resignation                                                 4
Record Date                                                 4
Change in Auditors                                          4
Reasons for Delayed/Non-submission of Financial Results     4
Cessation                                                  

In [32]:
print("Unmapped announcements by symbol:")

display(
    still_unmapped["symbol"]
    .value_counts()
    .head(50)
)

Unmapped announcements by symbol:


symbol
VALECHAENG    11
KUNDANMM      10
TALWALKARS    10
GAMMONIND      9
MELSTAR        8
XLENERGY       8
SRPL           7
QUINTEGRA      7
ARSSINFRA      7
MBECL          7
SECURCRED      6
OMKARCHEM      6
SHANKESH       5
FCONSUMER      5
SKIL           5
JBFIND         5
BLUEBLENDS     5
GANGOTRI       5
AUGMONT        5
BALLARPUR      5
AJRINFRA       5
ABHISHEK       5
KSERASERA      5
WINSOME        5
KSOILS         4
ATCOM          4
LEEL           4
GBGLOBAL       4
PUNJLLOYD      4
HORIZONIND     3
INDLMETER      3
ATNINTER       3
NITINFIRE      3
RMCL           3
DHARSUGAR      3
SABEVENTS      3
NAGAFERT       3
CANDC          3
BGLOBAL        3
GAJA           3
ALPSINDUS      3
LALITHAA       3
ERAINFRA       3
RUSHABEAR      2
INDUSFILA      2
GOENKA         2
CMICABLES      2
UNIVAFOODS     2
VALUEIND       2
VIDEOIND       2
Name: count, dtype: int64

In [38]:
news_df["final_company_id"] = (
    news_df["company_id"]
    .fillna(news_df["company_id_symbol"])
)

print(
    "Final mapped:",
    news_df["final_company_id"].notna().sum()
)

print(
    "Final unmapped:",
    news_df["final_company_id"].isna().sum()
)

Final mapped: 20775
Final unmapped: 249


In [33]:
cursor = conn.cursor(dictionary=True)

cursor.execute("DESCRIBE news")

news_schema = pd.DataFrame(cursor.fetchall())

display(news_schema)

cursor.close()
conn.close()

OperationalError: MySQL Connection not available.

In [34]:
import mysql.connector
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv("/Users/amit/Desktop/InvestIQ/backend/.env")

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    autocommit=True
)

print("MySQL connected:", conn.is_connected())

MySQL connected: True


In [35]:
cursor = conn.cursor(dictionary=True)

cursor.execute("DESCRIBE news")

news_schema = pd.DataFrame(cursor.fetchall())

display(news_schema)

cursor.close()

,Field,Type,Null,Key,Default,Extra
0,id,bigint,NO,PRI,None,auto_increment
1,company_id,int,NO,MUL,None,
2,symbol,varchar(50),NO,,None,
3,isin,varchar(20),NO,,None,
4,company_name,varchar(255),NO,,None,
5,headline,text,NO,,None,
6,content,longtext,YES,,None,
7,event_type,varchar(150),YES,,None,
8,importance,"enum('HIGH','MEDIUM','LOW')",YES,,None,
9,feed_type,varchar(100),NO,,None,


True

In [36]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    autocommit=True
)

cursor = conn.cursor()

cursor.execute("""
    ALTER TABLE news
        ADD COLUMN content LONGTEXT NULL AFTER headline,
        ADD COLUMN event_type VARCHAR(150) NULL AFTER content,
        ADD COLUMN nse_seq_id BIGINT NULL AFTER source,
        ADD COLUMN attachment_url TEXT NULL AFTER nse_seq_id,
        ADD COLUMN has_xbrl BOOLEAN NULL AFTER attachment_url
""")

print("News table updated successfully.")

cursor.close()
conn.close()

ProgrammingError: 1060 (42S21): Duplicate column name 'content'

In [37]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor(dictionary=True)

cursor.execute("DESCRIBE news")

display(pd.DataFrame(cursor.fetchall()))

cursor.close()
conn.close()

,Field,Type,Null,Key,Default,Extra
0,id,bigint,NO,PRI,None,auto_increment
1,company_id,int,NO,MUL,None,
2,symbol,varchar(50),NO,,None,
3,isin,varchar(20),NO,,None,
4,company_name,varchar(255),NO,,None,
5,headline,text,NO,,None,
6,content,longtext,YES,,None,
7,event_type,varchar(150),YES,,None,
8,importance,"enum('HIGH','MEDIUM','LOW')",YES,,None,
9,feed_type,varchar(100),NO,,None,


In [38]:
cursor = conn.cursor()

cursor.execute("""
    ALTER TABLE news
    ADD UNIQUE KEY unique_nse_seq (
        source,
        nse_seq_id
    )
""")

print("Unique NSE constraint added.")

cursor.close()

OperationalError: MySQL Connection not available.

In [39]:
import mysql.connector
import os
from dotenv import load_dotenv

load_dotenv("/Users/amit/Desktop/InvestIQ/backend/.env")

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    autocommit=True
)

cursor = conn.cursor()

print("Connected:", conn.is_connected())

# Check whether the unique key already exists
cursor.execute("""
    SHOW INDEX FROM news
    WHERE Key_name = 'unique_nse_seq'
""")

existing_index = cursor.fetchall()

if existing_index:
    print("unique_nse_seq already exists.")
else:
    cursor.execute("""
        ALTER TABLE news
        ADD UNIQUE KEY unique_nse_seq (source, nse_seq_id)
    """)
    print("unique_nse_seq added successfully.")

cursor.close()
conn.close()

Connected: True
unique_nse_seq already exists.


In [46]:
display(
    investiq_news[
        [
            "symbol",
            "company_name",
            "event_type",
            "attachment_text"
        ]
    ].head(20).to_string(index=False)
)

NameError: name 'investiq_news' is not defined

In [40]:
# Recreate the final InvestIQ news dataset

news_df["final_company_id"] = (
    news_df["company_id"]
    .fillna(news_df["company_id_symbol"])
)

investiq_news = news_df[
    news_df["final_company_id"].notna()
].copy()

investiq_news["final_company_id"] = (
    investiq_news["final_company_id"]
    .astype(int)
)

print("Total NSE announcements:", len(news_df))
print("InvestIQ news records:", len(investiq_news))
print(
    "Excluded unmapped records:",
    news_df["final_company_id"].isna().sum()
)


Total NSE announcements: 21024
InvestIQ news records: 20775
Excluded unmapped records: 249


In [41]:
display(
    investiq_news[
        [
            "symbol",
            "company_name",
            "event_type",
            "attachment_text"
        ]
    ].head(20).to_string(index=False)
)

'    symbol                           company_name                      event_type                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              attachment_text\nMUTHOOTFIN                Muthoot Finance Limited            Shareholders meeting                      

In [42]:
investiq_news["text_length"] = (
    investiq_news["attachment_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

display(investiq_news["text_length"].describe())

count    20775.000000
mean       133.343442
std         90.059093
min         13.000000
25%         83.000000
50%        112.000000
75%        157.000000
max       2938.000000
Name: text_length, dtype: float64

In [43]:
# Prepare final NSE dataset for MySQL ingestion

nse_insert_df = investiq_news.copy()

# Use the final mapped company ID
nse_insert_df["company_id"] = (
    nse_insert_df["final_company_id"]
    .astype(int)
)

# NSE announcement text
nse_insert_df["content"] = (
    nse_insert_df["attachment_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# For now, use the NSE event description as headline.
# We can later generate better headlines with an NLP layer.
nse_insert_df["headline"] = (
    nse_insert_df["event_type"]
    .fillna("NSE Announcement")
    .astype(str)
    .str.strip()
)

# Source
nse_insert_df["source"] = "NSE"

# Feed type tells us this came from the historical API
nse_insert_df["feed_type"] = "historical_announcement"

# Event date
nse_insert_df["event_date"] = pd.to_datetime(
    nse_insert_df["published_at"],
    format="%d-%b-%Y %H:%M:%S",
    errors="coerce"
).dt.date

# Published timestamp
nse_insert_df["published_at"] = pd.to_datetime(
    nse_insert_df["published_at"],
    format="%d-%b-%Y %H:%M:%S",
    errors="coerce"
)

# URL: use attachment URL
nse_insert_df["url"] = (
    nse_insert_df["attachment_url"]
    .fillna("")
    .astype(str)
)

# Keep only the DB fields
nse_insert_df = nse_insert_df[
    [
        "company_id",
        "symbol",
        "isin",
        "company_name",
        "headline",
        "content",
        "event_type",
        "feed_type",
        "published_at",
        "event_date",
        "url",
        "source",
        "nse_seq_id",
        "attachment_url",
        "hasXbrl"
    ]
].copy()

print("Records ready:", len(nse_insert_df))

display(nse_insert_df.head(10))

Records ready: 20775


,company_id,symbol,isin,company_name,headline,content,event_type,feed_type,published_at,event_date,url,source,nse_seq_id,attachment_url,hasXbrl
0,1500,MUTHOOTFIN,INE414G01012,Muthoot Finance Limited,Shareholders meeting,Muthoot Finance Limited has informed the Excha...,Shareholders meeting,historical_announcement,2026-08-31 23:48:18,2026-08-31,https://nsearchives.nseindia.com/corporate/MUT...,NSE,106763396,https://nsearchives.nseindia.com/corporate/MUT...,True
1,798,GICRE,INE481Y01014,General Insurance Corporation of India,Shareholders meeting,General Insurance Corporation of India has inf...,Shareholders meeting,historical_announcement,2026-08-31 23:36:42,2026-08-31,https://nsearchives.nseindia.com/corporate/GIC...,NSE,106763385,https://nsearchives.nseindia.com/corporate/GIC...,True
2,1158,JUSTDIAL,INE599M01018,Just Dial Limited,Change in Director(s),Just Dial Limited has informed the Exchange re...,Change in Director(s),historical_announcement,2026-08-31 23:36:14,2026-08-31,https://nsearchives.nseindia.com/corporate/Dim...,NSE,106763384,https://nsearchives.nseindia.com/corporate/Dim...,True
3,1158,JUSTDIAL,INE599M01018,Just Dial Limited,Shareholders meeting,Just Dial Limited has submitted the Exchange a...,Shareholders meeting,historical_announcement,2026-08-31 23:31:38,2026-08-31,https://nsearchives.nseindia.com/corporate/chi...,NSE,106763382,https://nsearchives.nseindia.com/corporate/chi...,True
4,640,E2E,INE255Z01019,E2E Networks Limited,Press Release,E2E Networks Limited has informed the Exchange...,Press Release,historical_announcement,2026-08-31 23:27:56,2026-08-31,https://nsearchives.nseindia.com/corporate/E2E...,NSE,106763381,https://nsearchives.nseindia.com/corporate/E2E...,True
5,2294,TI,INE133E01013,Tilaknagar Industries Limited,Record Date,Tilaknagar Industries Limited has informed the...,Record Date,historical_announcement,2026-08-31 23:26:24,2026-08-31,https://nsearchives.nseindia.com/corporate/gth...,NSE,106763379,https://nsearchives.nseindia.com/corporate/gth...,True
6,798,GICRE,INE481Y01014,General Insurance Corporation of India,Copy of Newspaper Publication,General Insurance Corporation of India has inf...,Copy of Newspaper Publication,historical_announcement,2026-08-31 23:24:00,2026-08-31,https://nsearchives.nseindia.com/corporate/GIC...,NSE,106763376,https://nsearchives.nseindia.com/corporate/GIC...,True
7,9,3IINFOLTD,INE748C01038,3i Infotech Limited,Change in Director(s),3i Infotech Limited has informed the Exchange ...,Change in Director(s),historical_announcement,2026-08-31 23:23:09,2026-08-31,https://nsearchives.nseindia.com/corporate/3II...,NSE,106763371,https://nsearchives.nseindia.com/corporate/3II...,True
8,2509,WELSPUNLIV,INE192B01031,Welspun Living Limited,General Updates,Welspun Living Limited has informed the Exchan...,General Updates,historical_announcement,2026-08-31 23:21:49,2026-08-31,https://nsearchives.nseindia.com/corporate/WEL...,NSE,106763368,https://nsearchives.nseindia.com/corporate/WEL...,True
9,1561,NIITMTS,INE342G01023,NIIT Learning Systems Limited,Other Restructuring,"Update of the MST Holding GmbH Germany, a step...",Other Restructuring,historical_announcement,2026-08-31 23:17:19,2026-08-31,https://nsearchives.nseindia.com/corporate/NLS...,NSE,106763366,https://nsearchives.nseindia.com/corporate/NLS...,True


In [44]:
print("Null values:")
display(
    nse_insert_df.isna().sum()
)

print("\nDuplicate NSE IDs:")
print(
    nse_insert_df["nse_seq_id"].duplicated().sum()
)

print("\nUnique NSE IDs:")
print(
    nse_insert_df["nse_seq_id"].nunique()
)

Null values:


company_id        0
symbol            0
isin              0
company_name      0
headline          0
content           0
event_type        0
feed_type         0
published_at      0
event_date        0
url               0
source            0
nse_seq_id        0
attachment_url    0
hasXbrl           0
dtype: int64


Duplicate NSE IDs:
0

Unique NSE IDs:
20775


In [45]:
import mysql.connector
import os

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    autocommit=False
)

cursor = conn.cursor()

insert_query = """
INSERT INTO news (
    company_id,
    symbol,
    isin,
    company_name,
    headline,
    content,
    event_type,
    feed_type,
    published_at,
    event_date,
    date_status,
    url,
    source,
    nse_seq_id,
    attachment_url,
    has_xbrl
)
VALUES (
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s
)
ON DUPLICATE KEY UPDATE
    content = VALUES(content),
    event_type = VALUES(event_type),
    published_at = VALUES(published_at),
    event_date = VALUES(event_date),
    date_status = VALUES(date_status),
    url = VALUES(url),
    attachment_url = VALUES(attachment_url),
    has_xbrl = VALUES(has_xbrl)
"""

rows = [
    tuple(row)
    for row in nse_insert_df[
        [
            "company_id",
            "symbol",
            "isin",
            "company_name",
            "headline",
            "content",
            "event_type",
            "feed_type",
            "published_at",
            "event_date",
            "url",
            "source",
            "nse_seq_id",
            "attachment_url",
            "hasXbrl"
        ]
    ].itertuples(index=False, name=None)
]

# Add date_status to every row
rows = [
    row[:10] + ("published",) + row[10:]
    for row in rows
]

cursor.executemany(insert_query, rows)

conn.commit()

print("Rows processed:", cursor.rowcount)

cursor.close()
conn.close()

Rows processed: 0


In [46]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT COUNT(*)
    FROM news
    WHERE source = 'NSE'
      AND feed_type = 'historical_announcement'
""")

print("NSE announcements in DB:", cursor.fetchone()[0])

cursor.execute("""
    SELECT
        MIN(published_at),
        MAX(published_at),
        COUNT(DISTINCT company_id),
        COUNT(DISTINCT nse_seq_id)
    FROM news
    WHERE source = 'NSE'
      AND feed_type = 'historical_announcement'
""")

print(cursor.fetchone())

cursor.close()
conn.close()

NSE announcements in DB: 20775
(datetime.datetime(2026, 8, 1, 8, 11, 24), datetime.datetime(2026, 8, 31, 23, 48, 18), 2265, 20775)


In [47]:
import mysql.connector
import os
import pandas as pd

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT company_id) AS companies,
        COUNT(DISTINCT symbol) AS symbols,
        COUNT(DISTINCT nse_seq_id) AS unique_nse_ids,
        MIN(published_at) AS oldest,
        MAX(published_at) AS newest
    FROM news
    WHERE source = 'NSE'
      AND feed_type = 'historical_announcement'
""")

result = cursor.fetchone()

print("Total NSE news:", result[0])
print("Companies:", result[1])
print("Symbols:", result[2])
print("Unique NSE IDs:", result[3])
print("Oldest:", result[4])
print("Newest:", result[5])

cursor.close()
conn.close()

Total NSE news: 20775
Companies: 2265
Symbols: 2266
Unique NSE IDs: 20775
Oldest: 2026-08-01 08:11:24
Newest: 2026-08-31 23:48:18


In [48]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

query = """
SELECT
    id,
    company_id,
    symbol,
    company_name,
    headline,
    LEFT(content, 250) AS content_preview,
    event_type,
    published_at,
    nse_seq_id
FROM news
WHERE source = 'NSE'
  AND feed_type = 'historical_announcement'
ORDER BY published_at DESC
LIMIT 10
"""

sample_df = pd.read_sql(query, conn)

display(sample_df)

conn.close()

/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_92867/3522386781.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sample_df = pd.read_sql(query, conn)


,id,company_id,symbol,company_name,headline,content_preview,event_type,published_at,nse_seq_id
0,265,1500,MUTHOOTFIN,Muthoot Finance Limited,Shareholders meeting,Muthoot Finance Limited has informed the Excha...,Shareholders meeting,2026-08-31 23:48:18,106763396
1,266,798,GICRE,General Insurance Corporation of India,Shareholders meeting,General Insurance Corporation of India has inf...,Shareholders meeting,2026-08-31 23:36:42,106763385
2,267,1158,JUSTDIAL,Just Dial Limited,Change in Director(s),Just Dial Limited has informed the Exchange re...,Change in Director(s),2026-08-31 23:36:14,106763384
3,268,1158,JUSTDIAL,Just Dial Limited,Shareholders meeting,Just Dial Limited has submitted the Exchange a...,Shareholders meeting,2026-08-31 23:31:38,106763382
4,269,640,E2E,E2E Networks Limited,Press Release,E2E Networks Limited has informed the Exchange...,Press Release,2026-08-31 23:27:56,106763381
5,270,2294,TI,Tilaknagar Industries Limited,Record Date,Tilaknagar Industries Limited has informed the...,Record Date,2026-08-31 23:26:24,106763379
6,271,798,GICRE,General Insurance Corporation of India,Copy of Newspaper Publication,General Insurance Corporation of India has inf...,Copy of Newspaper Publication,2026-08-31 23:24:00,106763376
7,272,9,3IINFOLTD,3i Infotech Limited,Change in Director(s),3i Infotech Limited has informed the Exchange ...,Change in Director(s),2026-08-31 23:23:09,106763371
8,273,2509,WELSPUNLIV,Welspun Living Limited,General Updates,Welspun Living Limited has informed the Exchan...,General Updates,2026-08-31 23:21:49,106763368
9,274,1561,NIITMTS,NIIT Learning Systems Limited,Other Restructuring,"Update of the MST Holding GmbH Germany, a step...",Other Restructuring,2026-08-31 23:17:19,106763366


In [49]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

query = """
SELECT
    event_type,
    COUNT(*) AS count
FROM news
WHERE source = 'NSE'
  AND feed_type = 'historical_announcement'
GROUP BY event_type
ORDER BY count DESC
"""

event_df = pd.read_sql(query, conn)

display(event_df.head(30))

conn.close()

/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_92867/2823220855.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  event_df = pd.read_sql(query, conn)


,event_type,count
0,Analysts/Institutional Investor Meet/Con. Call...,3442
1,Copy of Newspaper Publication,3043
2,General Updates,2185
3,Outcome of Board Meeting,2129
4,Shareholders meeting,2002
5,Updates,1394
6,Press Release,966
7,Investor Presentation,724
8,Record Date,498
9,Appointment,497


In [50]:
event_importance = {
    # HIGH IMPACT
    "Acquisition": "HIGH",
    "Bagging/Receiving of orders/contracts": "HIGH",
    "Credit Rating": "HIGH",
    "Change in Management": "HIGH",
    "Change in Auditors": "HIGH",
    "Outcome of Board Meeting": "HIGH",
    "Pendency of Litigation(s)/dispute(s) or the outcome": "HIGH",
    "Action(s) taken or orders passed": "HIGH",
    "Corporate Insolvency Resolution Process": "HIGH",
    "Allotment of Securities": "HIGH",
    "Disclosure under SEBI Takeover Regulations": "HIGH",
    "Disclosure of material issue": "HIGH",
    "Cessation": "HIGH",
    "Resignation": "HIGH",
    "Resignation of Director/KMP/SMP": "HIGH",

    # MEDIUM IMPACT
    "Appointment": "MEDIUM",
    "Change in Director(s)": "MEDIUM",
    "ESOP/ESOS/ESPS": "MEDIUM",
    "Monitoring Agency Report": "MEDIUM",
    "Statement of deviation(s) or variation(s) under": "MEDIUM",
    "Shareholders meeting": "MEDIUM",
    "Record Date": "MEDIUM",
    "Press Release": "MEDIUM",
    "Investor Presentation": "MEDIUM",
    "General Updates": "MEDIUM",
    "Updates": "MEDIUM",

    # LOW / ROUTINE
    "Copy of Newspaper Publication": "LOW",
    "Analysts/Institutional Investor Meet/Con. Call Updates": "LOW",
    "Trading Window": "LOW",
    "Price movement": "LOW",
    "Spurt in Volume": "LOW",
}

In [51]:
def classify_event_importance(event_type):
    if not isinstance(event_type, str):
        return "LOW"

    event = event_type.lower().strip()

    high_keywords = [
        "acquisition",
        "order",
        "contract",
        "credit rating",
        "change in management",
        "change in auditors",
        "outcome of board meeting",
        "litigation",
        "dispute",
        "insolvency",
        "allotment of securities",
        "takeover",
        "material issue",
        "cessation",
        "resignation",
    ]

    medium_keywords = [
        "appointment",
        "change in director",
        "esop",
        "monitoring agency",
        "deviation",
        "shareholders meeting",
        "record date",
        "press release",
        "investor presentation",
        "general updates",
        "updates",
    ]

    low_keywords = [
        "newspaper publication",
        "analysts",
        "institutional investor meet",
        "con. call",
        "trading window",
        "price movement",
        "spurt in volume",
    ]

    if any(keyword in event for keyword in high_keywords):
        return "HIGH"

    if any(keyword in event for keyword in medium_keywords):
        return "MEDIUM"

    if any(keyword in event for keyword in low_keywords):
        return "LOW"

    return "LOW"

In [52]:
event_df["importance"] = event_df["event_type"].apply(
    classify_event_importance
)

display(
    event_df.sort_values(
        ["importance", "count"],
        ascending=[True, False]
    )
)

,event_type,count,importance
3,Outcome of Board Meeting,2129,HIGH
10,Change in Management,336,HIGH
12,Resignation of Director/KMP/SMP,253,HIGH
15,Resignation,161,HIGH
16,Credit Rating,155,HIGH
...,...,...,...
20,Statement of deviation(s) or variation(s) unde...,125,MEDIUM
39,Committee Meeting Updates,38,MEDIUM
47,Monthly Business Updates,22,MEDIUM
61,Press Release (Revised),10,MEDIUM


In [53]:
print(
    event_df.groupby("importance")["count"]
    .sum()
    .sort_values(ascending=False)
)

importance
MEDIUM    12550
HIGH       4133
LOW        4092
Name: count, dtype: int64


In [54]:
low_events = event_df[
    event_df["importance"] == "LOW"
].sort_values("count", ascending=False)

display(low_events)

,event_type,count,importance
1,Copy of Newspaper Publication,3043,LOW
22,Spurt in Volume,95,LOW
26,Trading Window,62,LOW
27,Price movement,61,LOW
30,Dividend,57,LOW
...,...,...,...
104,Redemption,1,LOW
103,One time settlement,1,LOW
102,Increase in Authorised Capital,1,LOW
101,Amendment(s),1,LOW


In [55]:
print("LOW event types:", len(low_events))
print("LOW announcements:", low_events["count"].sum())

LOW event types: 73
LOW announcements: 4092


In [56]:
low_events = event_df[
    event_df["importance"] == "LOW"
].sort_values("count", ascending=False)

display(
    low_events[
        ["event_type", "count"]
    ].to_string(index=False)
)

'                                                                                 event_type  count\n                                                              Copy of Newspaper Publication   3043\n                                                                            Spurt in Volume     95\n                                                                             Trading Window     62\n                                                                             Price movement     61\n                                                                                   Dividend     57\n                                                               Integrated Filing- Financial     56\n                                                                       Amendment to AOA/MOA     56\n                                                                                 Agreements     51\n                                                                          News Verification     40\

In [57]:
event_df["importance_v2"] = ...

In [58]:
high_events = {
    "Acquisition",
    "Amalgamation/Merger",
    "One time settlement",
    "Corporate Debt Restructuring",
    "Fraud/Default/Arrest",
    "Initiation of Forensic Audit",
    "Disruption of Operations",
    "Closure of operations",
    "Granting/withdrawal/surrender/cancellation/suspension of key licenses/ regulatory approvals",
    "Sale or disposal",
    "Diversification/Disinvestment",
    "Capacity addition",
    "Commencement of commercial production/operations",
    "Product launch",
    "Qualified Institutional Placement",
    "Preferential issue",
    "Issue of Securities",
    "Rights Issue",
    "Buyback",
    "Public Announcement - Buyback of Shares",
    "Public Announcement-Open Offer",
    "Offer for sale",
    "Giving guarantees/indemnity/ becoming a surety for third party",
    "Utilisation of Funds",
    "Reasons for Delayed/Non-submission of Financial Results",
    "Clarification - Financial Results",
    "Reply to Clarification- Financial results",
    "Suspension of Trading",
    "Voluntary Delisting",
    "Demerger",
    "Scheme of Arrangement",
    "Other Restructuring",
    "Rescission/termination(s)",
    "Strikes/Lockouts/Disturbances",
    "Adoption of new line(s) of business",
    "Arrangements for strategic, technical, manufacturing, or marketing tie up",
    "Memorandum of Understanding/Agreements",
    "Agreements",
    "Trading Plan under PIT"
}

medium_events = {
    "Dividend",
    "Integrated Filing- Financial",
    "Amendment to AOA/MOA",
    "Options to purchase securities",
    "Change in Company Secretary/Compliance Officer",
    "Retirement",
    "Demise",
    "Structural Digital Database",
    "Certificate under SEBI (Depositories and Participants) Regulations, 2018",
    "Registrar & Share Transfer Agent Update",
    "Forfeiture",
    "Stock split",
    "Bonus",
    "Conversion",
    "Closure of Buy Back",
    "Post Buyback Public Announcement",
    "Redemption",
    "Increase in Authorised Capital",
    "Address Change",
    "Name Change",
    "Name and Symbol Change",
    "Effect(s) on listed entity due to changed regulatory framework applicable",
    "Delay/default in the payment of fines/penalties/dues etc. to authority",
    "Amendment(s)",
    "Addendum",
    "Corrigendum",
    "Rumour Verification - Regulation 30(11)",
    "News Verification"
}

low_events = {
    "Copy of Newspaper Publication",
    "Spurt in Volume",
    "Trading Window",
    "Price movement",
    "Integrated Filing- Governance",
    "Others",
    "Loss of Share Certificates"
}

def classify_event_v2(event_type):
    if event_type in high_events:
        return "HIGH"
    elif event_type in medium_events:
        return "MEDIUM"
    elif event_type in low_events:
        return "LOW"
    else:
        return "REVIEW"

In [59]:
event_df["importance_v2"] = event_df["event_type"].apply(
    classify_event_v2
)

print(
    event_df.groupby("importance_v2")["count"]
    .sum()
    .sort_values(ascending=False)
)

print("\nEvents needing review:")
display(
    event_df[
        event_df["importance_v2"] == "REVIEW"
    ].sort_values("count", ascending=False)
)

importance_v2
REVIEW    16554
LOW        3270
HIGH        537
MEDIUM      414
Name: count, dtype: int64

Events needing review:


,event_type,count,importance,importance_v2
0,Analysts/Institutional Investor Meet/Con. Call...,3442,MEDIUM,REVIEW
2,General Updates,2185,MEDIUM,REVIEW
3,Outcome of Board Meeting,2129,HIGH,REVIEW
4,Shareholders meeting,2002,MEDIUM,REVIEW
5,Updates,1394,MEDIUM,REVIEW
6,Press Release,966,MEDIUM,REVIEW
7,Investor Presentation,724,MEDIUM,REVIEW
8,Record Date,498,MEDIUM,REVIEW
9,Appointment,497,MEDIUM,REVIEW
10,Change in Management,336,HIGH,REVIEW


In [60]:
event_df["event_type_clean"] = (
    event_df["event_type"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [61]:
def classify_event_v2(event_type):
    event_type = str(event_type).strip()

    if event_type in high_events:
        return "HIGH"

    elif event_type in medium_events:
        return "MEDIUM"

    elif event_type in low_events:
        return "LOW"

    else:
        return "REVIEW"

In [62]:
event_df["importance_v2"] = event_df["event_type_clean"].apply(
    classify_event_v2
)

In [63]:
print(
    event_df.groupby("importance_v2")["count"]
    .sum()
    .sort_values(ascending=False)
)

importance_v2
REVIEW    16552
LOW        3270
HIGH        537
MEDIUM      416
Name: count, dtype: int64


In [64]:
review_events = event_df[
    event_df["importance_v2"] == "REVIEW"
].sort_values("count", ascending=False)

display(
    review_events[
        ["event_type_clean", "count"]
    ]
)

,event_type_clean,count
0,Analysts/Institutional Investor Meet/Con. Call...,3442
2,General Updates,2185
3,Outcome of Board Meeting,2129
4,Shareholders meeting,2002
5,Updates,1394
6,Press Release,966
7,Investor Presentation,724
8,Record Date,498
9,Appointment,497
10,Change in Management,336


In [65]:
import re

def classify_event_v3(event_type):
    if not isinstance(event_type, str):
        return "REVIEW"

    event = re.sub(r"\s+", " ", event_type.strip().lower())

    # =========================
    # HIGH IMPORTANCE
    # =========================

    high_patterns = [
        # Business / strategic
        "acquisition",
        "amalgamation",
        "merger",
        "demerger",
        "scheme of arrangement",
        "other restructuring",
        "corporate debt restructuring",
        "one time settlement",
        "sale or disposal",
        "diversification/disinvestment",
        "adoption of new line",
        "strategic, technical, manufacturing",
        "memorandum of understanding",
        "mou",
        "agreement",
        "arrangement",

        # Orders / business growth
        "bagging/receiving of orders",
        "awarding of order",
        "order(s)/contract(s)",
        "capacity addition",
        "commencement of commercial production",
        "product launch",

        # Financial / capital
        "credit rating",
        "qualified institutional placement",
        "preferential issue",
        "issue of securities",
        "rights issue",
        "allotment of securities",
        "buyback",
        "open offer",
        "offer for sale",
        "utilisation of funds",
        "increase in authorised capital",

        # Management
        "change in management",
        "resignation of director",
        "resignation of statutory auditor",
        "resignation",
        "cessation",

        # Legal / regulatory
        "litigation",
        "dispute",
        "insolvency",
        "forensic audit",
        "fraud/default/arrest",
        "fraud",
        "default",
        "arrest",
        "key licenses",
        "regulatory approvals",
        "orders passed",
        "action(s) initiated",
        "action(s) taken",
        "fines/penalties/dues",
        "suspension of trading",
        "voluntary delisting",

        # Operational risk
        "disruption of operations",
        "closure of operations",
        "strikes/lockouts/disturbances",
    ]

    # =========================
    # LOW IMPORTANCE
    # =========================

    low_patterns = [
        "copy of newspaper publication",
        "spurt in volume",
        "trading window",
        "price movement",
        "loss of share certificates",
        "integrated filing- governance",
        "integrated filing - governance",
        "structural digital database",
    ]

    # =========================
    # MEDIUM IMPORTANCE
    # =========================

    medium_patterns = [
        "dividend",
        "integrated filing- financial",
        "integrated filing - financial",
        "amendment to aoa/moa",
        "options to purchase securities",
        "change in company secretary",
        "compliance officer",
        "retirement",
        "demise",
        "record date",
        "revised record date",
        "appointment",
        "change in director",
        "esop",
        "monitoring agency",
        "deviation",
        "shareholders meeting",
        "press release",
        "investor presentation",
        "general updates",
        "monthly business updates",
        "committee meeting updates",
        "name change",
        "address change",
        "name and symbol change",
        "certificate under sebi",
        "registrar & share transfer agent",
        "forfeiture",
        "stock split",
        "bonus",
        "conversion",
        "redemption",
        "closure of buy back",
        "post buyback public announcement",
        "rumour verification",
        "news verification",
        "corrigendum",
        "addendum",
        "amendment",
        "effect(s) on listed entity",
    ]

    # Check HIGH first
    if any(pattern in event for pattern in high_patterns):
        return "HIGH"

    # Then LOW
    if any(pattern in event for pattern in low_patterns):
        return "LOW"

    # Then MEDIUM
    if any(pattern in event for pattern in medium_patterns):
        return "MEDIUM"

    return "REVIEW"

In [66]:
event_df["importance_v3"] = event_df["event_type_clean"].apply(
    classify_event_v3
)

In [67]:
print(
    event_df.groupby("importance_v3")["count"]
    .sum()
    .sort_values(ascending=False)
)

importance_v3
MEDIUM    8105
REVIEW    7303
LOW       3279
HIGH      2088
Name: count, dtype: int64


In [68]:
review_events = event_df[
    event_df["importance_v3"] == "REVIEW"
].sort_values("count", ascending=False)

print("REVIEW event types:", len(review_events))
print("REVIEW announcements:", review_events["count"].sum())

display(
    review_events[
        ["event_type_clean", "count"]
    ].to_string(index=False)
)

REVIEW event types: 13
REVIEW announcements: 7303


'                                              event_type_clean  count\n        Analysts/Institutional Investor Meet/Con. Call Updates   3442\n                                      Outcome of Board Meeting   2129\n                                                       Updates   1394\n                                            Change in Auditors    125\n                    Disclosure under SEBI Takeover Regulations     60\n                                  Disclosure of material issue     59\n                             Clarification - Financial Results     31\nGiving guarantees/indemnity/ becoming a surety for third party     20\n                     Reply to Clarification- Financial results     16\n       Reasons for Delayed/Non-submission of Financial Results     10\n                                        Trading Plan under PIT      8\n                                     Rescission/termination(s)      6\n                                                        Others      3'

In [69]:
high_patterns += [
    "outcome of board meeting",
    "change in auditors",
    "disclosure under sebi takeover regulations",
    "disclosure of material issue",
    "clarification - financial results",
    "reply to clarification- financial results",
    "reasons for delayed/non-submission of financial results",
    "giving guarantees/indemnity",
    "trading plan under pit",
    "rescission/termination"
]

medium_patterns += [
    "updates"
]

low_patterns += [
    "analysts/institutional investor meet/con. call updates",
    "others"
]

NameError: name 'high_patterns' is not defined

In [70]:
import re

def classify_event_v4(event_type):
    if not isinstance(event_type, str):
        return "REVIEW"

    event = re.sub(r"\s+", " ", event_type.strip().lower())

    # =========================
    # HIGH IMPORTANCE
    # =========================
    high_patterns = [
        # Board / management
        "outcome of board meeting",
        "change in management",
        "change in auditors",
        "resignation",
        "cessation",

        # Business / strategic
        "acquisition",
        "amalgamation",
        "merger",
        "demerger",
        "scheme of arrangement",
        "other restructuring",
        "corporate debt restructuring",
        "one time settlement",
        "sale or disposal",
        "diversification/disinvestment",
        "adoption of new line",
        "strategic, technical, manufacturing",
        "memorandum of understanding",
        "agreement",
        "arrangement",

        # Orders / operations
        "bagging/receiving of orders",
        "awarding of order",
        "order(s)/contract(s)",
        "capacity addition",
        "commencement of commercial production",
        "product launch",
        "disruption of operations",
        "closure of operations",
        "strikes/lockouts/disturbances",

        # Financial / capital
        "credit rating",
        "qualified institutional placement",
        "preferential issue",
        "issue of securities",
        "allotment of securities",
        "rights issue",
        "buyback",
        "open offer",
        "offer for sale",
        "utilisation of funds",
        "increase in authorised capital",

        # Legal / regulatory
        "litigation",
        "dispute",
        "insolvency",
        "forensic audit",
        "fraud/default/arrest",
        "fraud",
        "default",
        "arrest",
        "key licenses",
        "regulatory approvals",
        "orders passed",
        "action(s) initiated",
        "action(s) taken",
        "fines/penalties/dues",
        "suspension of trading",
        "voluntary delisting",

        # Financial-result issues
        "clarification - financial results",
        "reply to clarification- financial results",
        "reasons for delayed/non-submission of financial results",

        # SEBI / material disclosure
        "disclosure under sebi takeover regulations",
        "disclosure of material issue",

        # Guarantees / obligations
        "giving guarantees/indemnity",

        # Trading plan
        "trading plan under pit",

        # Termination
        "rescission/termination"
    ]

    # =========================
    # MEDIUM IMPORTANCE
    # =========================
    medium_patterns = [
        "dividend",
        "integrated filing- financial",
        "integrated filing - financial",
        "amendment to aoa/moa",
        "options to purchase securities",
        "change in company secretary",
        "compliance officer",
        "retirement",
        "demise",
        "record date",
        "revised record date",
        "appointment",
        "change in director",
        "esop",
        "monitoring agency",
        "deviation",
        "shareholders meeting",
        "press release",
        "investor presentation",
        "general updates",
        "updates",
        "monthly business updates",
        "committee meeting updates",
        "name change",
        "address change",
        "name and symbol change",
        "certificate under sebi",
        "registrar & share transfer agent",
        "forfeiture",
        "stock split",
        "bonus",
        "conversion",
        "redemption",
        "closure of buy back",
        "post buyback public announcement",
        "rumour verification",
        "news verification",
        "corrigendum",
        "addendum",
        "amendment",
        "effect(s) on listed entity"
    ]

    # =========================
    # LOW IMPORTANCE
    # =========================
    low_patterns = [
        "copy of newspaper publication",
        "spurt in volume",
        "trading window",
        "price movement",
        "loss of share certificates",
        "integrated filing- governance",
        "integrated filing - governance",
        "structural digital database",
        "analysts/institutional investor meet/con. call updates",
        "others"
    ]

    # HIGH first
    if any(pattern in event for pattern in high_patterns):
        return "HIGH"

    # LOW second
    if any(pattern in event for pattern in low_patterns):
        return "LOW"

    # MEDIUM third
    if any(pattern in event for pattern in medium_patterns):
        return "MEDIUM"

    return "REVIEW"

In [71]:
event_df["importance_v4"] = event_df["event_type_clean"].apply(
    classify_event_v4
)

In [72]:
print(
    event_df.groupby("importance_v4")["count"]
    .sum()
    .sort_values(ascending=False)
)

importance_v4
MEDIUM    9508
LOW       6724
HIGH      4543
Name: count, dtype: int64


In [73]:
review_events = event_df[
    event_df["importance_v4"] == "REVIEW"
].sort_values("count", ascending=False)

print("REVIEW event types:", len(review_events))
print("REVIEW announcements:", review_events["count"].sum())

display(
    review_events[
        ["event_type_clean", "count"]
    ]
)

REVIEW event types: 0
REVIEW announcements: 0


,event_type_clean,count


In [74]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("SHOW COLUMNS FROM news LIKE 'importance'")

result = cursor.fetchone()

print(result)

cursor.close()
conn.close()

('importance', "enum('HIGH','MEDIUM','LOW')", 'YES', '', None, '')


In [76]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("""
    ALTER TABLE news
    ADD COLUMN importance ENUM('HIGH', 'MEDIUM', 'LOW') NULL
    AFTER event_type
""")

conn.commit()

print("importance column added")

cursor.close()
conn.close()

ProgrammingError: 1060 (42S21): Duplicate column name 'importance'

In [77]:
event_df["importance_v4"]

0         LOW
1         LOW
2      MEDIUM
3        HIGH
4      MEDIUM
        ...  
105      HIGH
106    MEDIUM
107       LOW
108      HIGH
109    MEDIUM
Name: importance_v4, Length: 110, dtype: object

In [78]:
importance_map = (
    event_df
    .set_index("event_type_clean")["importance_v4"]
    .to_dict()
)

investiq_news["event_type_clean"] = (
    investiq_news["event_type"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

investiq_news["importance"] = (
    investiq_news["event_type_clean"]
    .map(importance_map)
)

print(
    investiq_news["importance"].value_counts(dropna=False)
)

importance
MEDIUM    9508
LOW       6724
HIGH      4543
Name: count, dtype: int64


In [79]:
print(
    "Missing importance:",
    investiq_news["importance"].isna().sum()
)

Missing importance: 0


In [80]:
import mysql.connector
import os

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    autocommit=False
)

cursor = conn.cursor()

update_query = """
UPDATE news
SET importance = %s
WHERE source = 'NSE'
  AND feed_type = 'historical_announcement'
  AND nse_seq_id = %s
"""

rows = [
    (row.importance, int(row.nse_seq_id))
    for row in investiq_news[
        ["importance", "nse_seq_id"]
    ].itertuples(index=False)
]

cursor.executemany(update_query, rows)

conn.commit()

print("Rows updated:", cursor.rowcount)

cursor.close()
conn.close()

Rows updated: 20775


In [81]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

query = """
SELECT
    importance,
    COUNT(*) AS count
FROM news
WHERE source = 'NSE'
  AND feed_type = 'historical_announcement'
GROUP BY importance
ORDER BY
    FIELD(importance, 'HIGH', 'MEDIUM', 'LOW')
"""

db_importance_df = pd.read_sql(query, conn)

display(db_importance_df)

conn.close()

/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_92867/231563556.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  db_importance_df = pd.read_sql(query, conn)


,importance,count
0,HIGH,4543
1,MEDIUM,9508
2,LOW,6724


In [82]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        COUNT(*) AS total,
        SUM(importance = 'HIGH') AS high_count,
        SUM(importance = 'MEDIUM') AS medium_count,
        SUM(importance = 'LOW') AS low_count,
        SUM(importance IS NULL) AS null_count
    FROM news
    WHERE source = 'NSE'
      AND feed_type = 'historical_announcement'
""")

print(cursor.fetchone())

cursor.close()
conn.close()

(20775, Decimal('4543'), Decimal('9508'), Decimal('6724'), Decimal('0'))


In [83]:
import mysql.connector
import os
import pandas as pd

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

query = """
SELECT
    id,
    company_id,
    symbol,
    isin,
    company_name,
    headline,
    content,
    event_type,
    feed_type,
    published_at,
    url,
    source,
    nse_seq_id
FROM news
WHERE source = 'NSE'
  AND feed_type != 'historical_announcement'
ORDER BY published_at DESC
"""

rss_db_df = pd.read_sql(query, conn)

conn.close()

print("RSS records in DB:", len(rss_db_df))
display(rss_db_df.head(10))

/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_92867/4090382548.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rss_db_df = pd.read_sql(query, conn)


RSS records in DB: 66


,id,company_id,symbol,isin,company_name,headline,content,event_type,feed_type,published_at,url,source,nse_seq_id
0,199,666,ELGIEQUIP,INE285A01027,Elgi Equipments Limited,Elgi Equipments Limited has informed the Excha...,None,None,online_announcement,2026-08-29 07:48:32,https://nsearchives.nseindia.com/corporate/ELG...,NSE,None
1,200,2070,SIGNPOST,INE0KGZ01021,Signpost India Limited,SIGNPOST INDIA LIMITED has informed the Exchan...,None,None,online_announcement,2026-08-29 06:32:55,https://nsearchives.nseindia.com/corporate/xbr...,NSE,None
2,201,2070,SIGNPOST,INE0KGZ01021,Signpost India Limited,Signpost India Limited has informed the Exchan...,None,None,online_announcement,2026-08-29 05:29:08,https://nsearchives.nseindia.com/corporate/SIG...,NSE,None
3,202,2070,SIGNPOST,INE0KGZ01021,Signpost India Limited,Signpost India Limited has informed the Exchan...,None,None,online_announcement,2026-08-29 05:25:44,https://nsearchives.nseindia.com/corporate/SIG...,NSE,None
4,203,2070,SIGNPOST,INE0KGZ01021,Signpost India Limited,Signpost India Limited has informed the Exchan...,None,None,online_announcement,2026-08-29 05:23:00,https://nsearchives.nseindia.com/corporate/SIG...,NSE,None
5,204,1400,MAXESTATES,INE03EI01018,Max Estates Limited,Max Estates Limited has informed the Exchange ...,None,None,online_announcement,2026-08-29 02:04:10,https://nsearchives.nseindia.com/corporate/xbr...,NSE,None
6,205,1400,MAXESTATES,INE03EI01018,Max Estates Limited,Max Estates Limited has informed the Exchange ...,None,None,online_announcement,2026-08-29 01:47:37,https://nsearchives.nseindia.com/corporate/xbr...,NSE,None
7,206,1400,MAXESTATES,INE03EI01018,Max Estates Limited,Max Estates Limited has informed the Exchange ...,None,None,online_announcement,2026-08-29 01:47:29,https://nsearchives.nseindia.com/corporate/xbr...,NSE,None
8,207,1400,MAXESTATES,INE03EI01018,Max Estates Limited,Max Estates Limited has informed the Exchange ...,None,None,online_announcement,2026-08-29 01:39:08,https://nsearchives.nseindia.com/corporate/xbr...,NSE,None
9,208,1400,MAXESTATES,INE03EI01018,Max Estates Limited,Max Estates Limited has informed the Exchange ...,None,None,online_announcement,2026-08-29 01:38:59,https://nsearchives.nseindia.com/corporate/xbr...,NSE,None


In [84]:
print(
    rss_db_df.groupby(["source", "feed_type"])
    .size()
    .sort_values(ascending=False)
)

source  feed_type                
NSE     online_announcement          28
        annual_report                20
        related_party_transaction    14
        financial_result              4
dtype: int64


In [85]:
print(
    "RSS records with NSE ID:",
    rss_db_df["nse_seq_id"].notna().sum()
)

print(
    "RSS records without NSE ID:",
    rss_db_df["nse_seq_id"].isna().sum()
)

RSS records with NSE ID: 0
RSS records without NSE ID: 66


In [86]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

nse_query = """
SELECT
    id,
    company_id,
    symbol,
    company_name,
    headline,
    content,
    event_type,
    published_at,
    event_date,
    url,
    nse_seq_id
FROM news
WHERE source = 'NSE'
  AND feed_type = 'historical_announcement'
"""

nse_db_df = pd.read_sql(nse_query, conn)

conn.close()

print("NSE historical records:", len(nse_db_df))

/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_92867/3735595186.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  nse_db_df = pd.read_sql(nse_query, conn)


NSE historical records: 20775


In [87]:
import re

def normalize_news_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [88]:
rss_db_df["text_norm"] = (
    rss_db_df["headline"].fillna("") + " " +
    rss_db_df["content"].fillna("")
).apply(normalize_news_text)

nse_db_df["text_norm"] = (
    nse_db_df["headline"].fillna("") + " " +
    nse_db_df["content"].fillna("")
).apply(normalize_news_text)

In [89]:
nse_text_lookup = {}

for idx, row in nse_db_df.iterrows():
    text = row["text_norm"]

    if text:
        nse_text_lookup.setdefault(text, []).append(idx)

rss_db_df["exact_text_match"] = rss_db_df["text_norm"].apply(
    lambda x: nse_text_lookup.get(x, [])
)

print(
    "RSS records with exact text match:",
    (rss_db_df["exact_text_match"].str.len() > 0).sum()
)

RSS records with exact text match: 0


In [90]:
nse_url_set = set(
    nse_db_df["url"]
    .dropna()
    .astype(str)
    .str.strip()
)

rss_db_df["url_match"] = rss_db_df["url"].astype(str).str.strip().isin(
    nse_url_set
)

print(
    "RSS records with matching URL:",
    rss_db_df["url_match"].sum()
)

RSS records with matching URL: 14


In [91]:
from rapidfuzz.fuzz import ratio

# Create a company-based lookup to avoid comparing
# every RSS record against all 20,775 NSE records.

nse_by_company = {
    company_id: group.copy()
    for company_id, group in nse_db_df.groupby("company_id")
}

matches = []

for _, rss in rss_db_df.iterrows():

    company_records = nse_by_company.get(rss["company_id"])

    if company_records is None:
        matches.append({
            "rss_id": rss["id"],
            "best_nse_id": None,
            "similarity": 0
        })
        continue

    rss_text = normalize_news_text(rss["headline"])

    best_score = 0
    best_nse_id = None

    for _, nse in company_records.iterrows():

        nse_text = normalize_news_text(nse["content"])

        if not rss_text or not nse_text:
            continue

        score = ratio(rss_text, nse_text)

        if score > best_score:
            best_score = score
            best_nse_id = nse["id"]

    matches.append({
        "rss_id": rss["id"],
        "best_nse_id": best_nse_id,
        "similarity": best_score
    })

rss_match_df = pd.DataFrame(matches)

display(
    rss_match_df.sort_values(
        "similarity",
        ascending=False
    ).head(20)
)

,rss_id,best_nse_id,similarity
4,203,1336.0,96.279070
2,201,1334.0,96.172249
22,221,1341.0,91.292876
21,220,1340.0,91.129032
25,224,1344.0,90.651558
3,202,1335.0,89.454545
24,223,1343.0,88.796680
19,218,1338.0,88.259109
27,226,1346.0,86.192469
26,225,1346.0,86.192469


In [92]:
rss_match_df = (
    rss_match_df
    .merge(
        rss_db_df[
            ["id", "symbol", "company_name", "published_at", "headline"]
        ],
        left_on="rss_id",
        right_on="id",
        how="left",
        suffixes=("", "_rss")
    )
    .merge(
        nse_db_df[
            ["id", "published_at", "event_type", "content", "nse_seq_id"]
        ],
        left_on="best_nse_id",
        right_on="id",
        how="left",
        suffixes=("", "_nse")
    )
)

display(
    rss_match_df[
        [
            "rss_id",
            "symbol",
            "published_at",
            "best_nse_id",
            "published_at_nse",
            "similarity",
            "event_type",
            "nse_seq_id"
        ]
    ].sort_values("similarity", ascending=False)
)

,rss_id,symbol,published_at,best_nse_id,published_at_nse,similarity,event_type,nse_seq_id
4,203,SIGNPOST,2026-08-29 05:23:00,1336.0,2026-08-29 05:23:00,96.279070,Updates,106761105.0
2,201,SIGNPOST,2026-08-29 05:29:08,1334.0,2026-08-29 05:29:08,96.172249,Updates,106761107.0
22,221,MAXESTATES,2026-08-29 00:19:46,1341.0,2026-08-29 00:19:46,91.292876,Outcome of Board Meeting,106761059.0
21,220,MAXESTATES,2026-08-29 00:22:03,1340.0,2026-08-29 00:22:03,91.129032,Press Release,106761060.0
25,224,MAXESTATES,2026-08-29 00:08:36,1344.0,2026-08-29 00:08:36,90.651558,Outcome of Board Meeting,106761056.0
...,...,...,...,...,...,...,...,...
45,264,SFML,2025-02-21 15:12:33,NaN,NaT,0.000000,NaN,NaN
44,263,LLOYDS,2025-04-25 21:12:12,NaN,NaT,0.000000,NaN,NaN
43,262,GANESHIN,2025-04-26 19:14:11,NaN,NaT,0.000000,NaN,NaN
42,261,QUESTLAB,2025-04-28 14:07:07,NaN,NaT,0.000000,NaN,NaN


In [93]:
# Add exact time difference between RSS and best NSE match

rss_match_df["published_at"] = pd.to_datetime(
    rss_match_df["published_at"],
    errors="coerce"
)

rss_match_df["published_at_nse"] = pd.to_datetime(
    rss_match_df["published_at_nse"],
    errors="coerce"
)

rss_match_df["time_diff_minutes"] = (
    (rss_match_df["published_at"] - rss_match_df["published_at_nse"])
    .abs()
    .dt.total_seconds()
    / 60
)

# Candidate duplicate rule
rss_match_df["duplicate_candidate"] = (
    (rss_match_df["best_nse_id"].notna()) &
    (rss_match_df["similarity"] >= 80) &
    (rss_match_df["time_diff_minutes"] <= 10)
)

print(
    "Duplicate candidates:",
    rss_match_df["duplicate_candidate"].sum()
)

display(
    rss_match_df[
        [
            "rss_id",
            "symbol",
            "published_at",
            "best_nse_id",
            "published_at_nse",
            "time_diff_minutes",
            "similarity",
            "event_type",
            "nse_seq_id",
            "duplicate_candidate"
        ]
    ]
    .sort_values(
        ["duplicate_candidate", "similarity"],
        ascending=[False, False]
    )
)

Duplicate candidates: 14


,rss_id,symbol,published_at,best_nse_id,published_at_nse,time_diff_minutes,similarity,event_type,nse_seq_id,duplicate_candidate
4,203,SIGNPOST,2026-08-29 05:23:00,1336.0,2026-08-29 05:23:00,0.0,96.279070,Updates,106761105.0,True
2,201,SIGNPOST,2026-08-29 05:29:08,1334.0,2026-08-29 05:29:08,0.0,96.172249,Updates,106761107.0,True
22,221,MAXESTATES,2026-08-29 00:19:46,1341.0,2026-08-29 00:19:46,0.0,91.292876,Outcome of Board Meeting,106761059.0,True
21,220,MAXESTATES,2026-08-29 00:22:03,1340.0,2026-08-29 00:22:03,0.0,91.129032,Press Release,106761060.0,True
25,224,MAXESTATES,2026-08-29 00:08:36,1344.0,2026-08-29 00:08:36,0.0,90.651558,Outcome of Board Meeting,106761056.0,True
...,...,...,...,...,...,...,...,...,...,...
51,250,AHIMSA,NaT,NaN,NaT,NaN,0.000000,NaN,NaN,False
52,243,PCCL,NaT,NaN,NaT,NaN,0.000000,NaN,NaN,False
58,237,SKP,NaT,NaN,NaT,NaN,0.000000,NaN,NaN,False
59,236,OMFURN,NaT,NaN,NaT,NaN,0.000000,NaN,NaN,False


In [94]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("""
    ALTER TABLE news
    ADD COLUMN is_duplicate TINYINT(1) NOT NULL DEFAULT 0
    AFTER importance
""")

cursor.execute("""
    ALTER TABLE news
    ADD COLUMN duplicate_of BIGINT NULL
    AFTER is_duplicate
""")

conn.commit()

print("Duplicate tracking columns added.")

cursor.close()
conn.close()

Duplicate tracking columns added.


In [95]:
duplicate_matches = rss_match_df[
    rss_match_df["duplicate_candidate"] == True
][
    ["rss_id", "best_nse_id"]
].copy()

print("Duplicates to mark:", len(duplicate_matches))

display(duplicate_matches)

Duplicates to mark: 14


,rss_id,best_nse_id
0,199,1333.0
2,201,1334.0
3,202,1335.0
4,203,1336.0
18,217,1337.0
19,218,1338.0
20,219,1339.0
21,220,1340.0
22,221,1341.0
23,222,1342.0


In [96]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    autocommit=False
)

cursor = conn.cursor()

update_query = """
UPDATE news
SET
    is_duplicate = 1,
    duplicate_of = %s
WHERE id = %s
"""

rows = [
    (int(row.best_nse_id), int(row.rss_id))
    for row in duplicate_matches.itertuples(index=False)
]

cursor.executemany(update_query, rows)

conn.commit()

print("RSS duplicates marked:", cursor.rowcount)

cursor.close()
conn.close()

RSS duplicates marked: 14


In [97]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        is_duplicate,
        COUNT(*)
    FROM news
    GROUP BY is_duplicate
    ORDER BY is_duplicate
""")

print(cursor.fetchall())

cursor.execute("""
    SELECT
        COUNT(*)
    FROM news
    WHERE is_duplicate = 1
      AND duplicate_of IS NOT NULL
""")

print("Linked duplicates:", cursor.fetchone()[0])

cursor.close()
conn.close()

[(0, 20827), (1, 14)]
Linked duplicates: 14


In [98]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

query = """
SELECT
    source,
    feed_type,
    COUNT(*) AS count
FROM news
GROUP BY source, feed_type
ORDER BY source, feed_type
"""

df = pd.read_sql(query, conn)

display(df)

conn.close()

/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_92867/2114766429.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,source,feed_type,count
0,NSE,annual_report,20
1,NSE,financial_result,4
2,NSE,historical_announcement,20775
3,NSE,online_announcement,28
4,NSE,related_party_transaction,14


In [99]:
conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        COUNT(*) AS total,
        SUM(is_duplicate = 1) AS duplicates,
        SUM(is_duplicate = 0) AS originals
    FROM news
""")

print(cursor.fetchone())

cursor.close()
conn.close()

(20841, Decimal('14'), Decimal('20827'))


In [100]:
display(
    rss_match_df[
        rss_match_df["best_nse_id"] == 1346
    ][
        [
            "rss_id",
            "symbol",
            "published_at",
            "best_nse_id",
            "published_at_nse",
            "similarity",
            "event_type",
            "nse_seq_id"
        ]
    ]
)

,rss_id,symbol,published_at,best_nse_id,published_at_nse,similarity,event_type,nse_seq_id
5,204,MAXESTATES,2026-08-29 02:04:10,1346.0,2026-08-29 00:02:45,67.811159,Outcome of Board Meeting,106761054.0
6,205,MAXESTATES,2026-08-29 01:47:37,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
7,206,MAXESTATES,2026-08-29 01:47:29,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
8,207,MAXESTATES,2026-08-29 01:39:08,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
9,208,MAXESTATES,2026-08-29 01:38:59,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
10,209,MAXESTATES,2026-08-29 01:38:47,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
11,210,MAXESTATES,2026-08-29 01:22:55,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
12,211,MAXESTATES,2026-08-29 01:22:43,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
13,212,MAXESTATES,2026-08-29 01:22:25,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0
14,213,MAXESTATES,2026-08-29 01:22:05,1346.0,2026-08-29 00:02:45,63.035019,Outcome of Board Meeting,106761054.0


In [101]:
import os
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

query = """
SELECT
    id,
    company_id,
    symbol,
    isin,
    company_name,
    headline,
    content,
    event_type,
    importance,
    published_at,
    event_date,
    url,
    source,
    nse_seq_id,
    attachment_url
FROM news
WHERE is_duplicate = 0
  AND importance IN ('HIGH', 'MEDIUM')
ORDER BY published_at DESC
"""

fingpt_df = pd.read_sql(query, conn)

conn.close()

print("FinGPT candidate rows:", len(fingpt_df))
display(fingpt_df.head())

/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_92867/4041844598.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fingpt_df = pd.read_sql(query, conn)


FinGPT candidate rows: 14051


,id,company_id,symbol,isin,company_name,headline,content,event_type,importance,published_at,event_date,url,source,nse_seq_id,attachment_url
0,265,1500,MUTHOOTFIN,INE414G01012,Muthoot Finance Limited,Shareholders meeting,Muthoot Finance Limited has informed the Excha...,Shareholders meeting,MEDIUM,2026-08-31 23:48:18,2026-08-31,https://nsearchives.nseindia.com/corporate/MUT...,NSE,106763396,https://nsearchives.nseindia.com/corporate/MUT...
1,266,798,GICRE,INE481Y01014,General Insurance Corporation of India,Shareholders meeting,General Insurance Corporation of India has inf...,Shareholders meeting,MEDIUM,2026-08-31 23:36:42,2026-08-31,https://nsearchives.nseindia.com/corporate/GIC...,NSE,106763385,https://nsearchives.nseindia.com/corporate/GIC...
2,267,1158,JUSTDIAL,INE599M01018,Just Dial Limited,Change in Director(s),Just Dial Limited has informed the Exchange re...,Change in Director(s),MEDIUM,2026-08-31 23:36:14,2026-08-31,https://nsearchives.nseindia.com/corporate/Dim...,NSE,106763384,https://nsearchives.nseindia.com/corporate/Dim...
3,268,1158,JUSTDIAL,INE599M01018,Just Dial Limited,Shareholders meeting,Just Dial Limited has submitted the Exchange a...,Shareholders meeting,MEDIUM,2026-08-31 23:31:38,2026-08-31,https://nsearchives.nseindia.com/corporate/chi...,NSE,106763382,https://nsearchives.nseindia.com/corporate/chi...
4,269,640,E2E,INE255Z01019,E2E Networks Limited,Press Release,E2E Networks Limited has informed the Exchange...,Press Release,MEDIUM,2026-08-31 23:27:56,2026-08-31,https://nsearchives.nseindia.com/corporate/E2E...,NSE,106763381,https://nsearchives.nseindia.com/corporate/E2E...


In [102]:
print("Importance distribution:")
display(
    fingpt_df["importance"]
    .value_counts()
    .to_frame("count")
)

print("\nEvent type distribution:")
display(
    fingpt_df["event_type"]
    .value_counts()
    .head(30)
    .to_frame("count")
)

Importance distribution:


,count
importance,
MEDIUM,9508
HIGH,4543



Event type distribution:


,count
event_type,
General Updates,2185
Outcome of Board Meeting,2129
Shareholders meeting,2002
Updates,1394
Press Release,966
Investor Presentation,724
Record Date,498
Appointment,497
Change in Management,336


In [103]:
def build_fingpt_text(row):
    company = str(row["company_name"]).strip()
    symbol = str(row["symbol"]).strip()
    event = str(row["event_type"]).strip()
    content = str(row["content"]).strip()

    return f"""
Company: {company}
Symbol: {symbol}
Event Type: {event}

Announcement:
{content}
""".strip()


fingpt_df["fingpt_text"] = fingpt_df.apply(
    build_fingpt_text,
    axis=1
)

display(
    fingpt_df[
        [
            "id",
            "symbol",
            "importance",
            "event_type",
            "fingpt_text"
        ]
    ].head(10)
)

,id,symbol,importance,event_type,fingpt_text
0,265,MUTHOOTFIN,MEDIUM,Shareholders meeting,Company: Muthoot Finance Limited\nSymbol: MUTH...
1,266,GICRE,MEDIUM,Shareholders meeting,Company: General Insurance Corporation of Indi...
2,267,JUSTDIAL,MEDIUM,Change in Director(s),Company: Just Dial Limited\nSymbol: JUSTDIAL\n...
3,268,JUSTDIAL,MEDIUM,Shareholders meeting,Company: Just Dial Limited\nSymbol: JUSTDIAL\n...
4,269,E2E,MEDIUM,Press Release,Company: E2E Networks Limited\nSymbol: E2E\nEv...
5,270,TI,MEDIUM,Record Date,Company: Tilaknagar Industries Limited\nSymbol...
6,272,3IINFOLTD,MEDIUM,Change in Director(s),Company: 3i Infotech Limited\nSymbol: 3IINFOLT...
7,273,WELSPUNLIV,MEDIUM,General Updates,Company: Welspun Living Limited\nSymbol: WELSP...
8,274,NIITMTS,HIGH,Other Restructuring,Company: NIIT Learning Systems Limited\nSymbol...
9,275,3IINFOLTD,MEDIUM,Shareholders meeting,Company: 3i Infotech Limited\nSymbol: 3IINFOLT...


In [104]:
pd.set_option("display.max_colwidth", 500)

sample_events = [
    "General Updates",
    "Updates",
    "Shareholders meeting",
    "Press Release",
    "Investor Presentation",
    "Outcome of Board Meeting",
    "Change in Management",
    "Bagging/Receiving of orders/contracts",
    "Credit Rating",
    "Acquisition"
]

for event in sample_events:
    print("\n" + "=" * 100)
    print(f"EVENT TYPE: {event}")
    print("=" * 100)

    sample = fingpt_df[
        fingpt_df["event_type"] == event
    ][
        ["id", "symbol", "company_name", "published_at", "content"]
    ].head(2)

    for _, row in sample.iterrows():
        print(f"\nID: {row['id']}")
        print(f"Company: {row['company_name']} ({row['symbol']})")
        print(f"Date: {row['published_at']}")
        print(f"Content:\n{row['content']}")


EVENT TYPE: General Updates

ID: 273
Company: Welspun Living Limited (WELSPUNLIV)
Date: 2026-08-31 23:21:49
Content:
Welspun Living Limited has informed the Exchange about Update on acquisition of 26% equity stake in Clean Max Dhyuthi Private Limited.

ID: 277
Company: Neogen Chemicals Limited (NEOGEN)
Date: 2026-08-31 23:10:04
Content:
Neogen Chemicals Limited has informed the Exchange about disclosure pursuant to regulation 30 (9) read with Schedule III of the SEBI LODR.

EVENT TYPE: Updates

ID: 287
Company: Onesource Specialty Pharma Limited (ONESOURCE)
Date: 2026-08-31 22:36:17
Content:
Onesource Specialty Pharma Limited has informed the Exchange regarding 'Intimation of Business Responsibility and Sustainability Report for FY 25-26'.

ID: 299
Company: Tilaknagar Industries Limited (TI)
Date: 2026-08-31 21:52:05
Content:
Tilaknagar Industries Limited has informed the Exchange regarding 'Intimation for Book Closure and Record Date for payment of Dividend'.

EVENT TYPE: Shareholder

In [105]:
fingpt_df["content_length"] = (
    fingpt_df["content"]
    .fillna("")
    .astype(str)
    .str.len()
)

display(
    fingpt_df.groupby("importance")["content_length"]
    .describe()
)

,count,mean,std,min,25%,50%,75%,max
importance,,,,,,,,
HIGH,4543.0,135.712305,113.533253,16.0,95.0,113.0,135.0,2938.0
MEDIUM,9508.0,145.206563,86.560671,13.0,94.0,128.0,175.0,2838.0


In [106]:
display(
    fingpt_df[
        ["id", "symbol", "event_type", "importance", "content_length"]
    ]
    .sort_values("content_length")
    .head(30)
)

,id,symbol,event_type,importance,content_length
2495,3802,NETWEB,Press Release,MEDIUM,13
12929,19374,UGROCAP,Press Release,MEDIUM,13
12156,18153,MODINATUR,General Updates,MEDIUM,14
8144,12106,MUKANDLTD,Shareholders meeting,MEDIUM,14
9674,14282,BASML,Shareholders meeting,MEDIUM,14
864,1406,RELIANCE,Updates,MEDIUM,15
1097,1779,ASAHIINDIA,General Updates,MEDIUM,15
10136,15037,DAMODARIND,Outcome of Board Meeting,HIGH,16
7769,11577,SHRINGARMS,Monitoring Agency Report,MEDIUM,19
1257,1981,SHIVAMILLS,Shareholders meeting,MEDIUM,20


In [107]:
import numpy as np

fingpt_df["content"] = fingpt_df["content"].fillna("").astype(str).str.strip()

fingpt_df["content_length"] = fingpt_df["content"].str.len()

fingpt_df["content_quality"] = np.select(
    [
        fingpt_df["content_length"] < 40,
        fingpt_df["content_length"] < 80,
        fingpt_df["content_length"] >= 80
    ],
    [
        "LOW",
        "MEDIUM",
        "GOOD"
    ],
    default="LOW"
)

display(
    fingpt_df["content_quality"]
    .value_counts()
    .to_frame("count")
)

,count
content_quality,
GOOD,12127
MEDIUM,1829
LOW,95


In [108]:
display(
    fingpt_df.groupby(
        ["importance", "content_quality"]
    ).size().unstack(fill_value=0)
)

content_quality,GOOD,LOW,MEDIUM
importance,,,
HIGH,3971,21,551
MEDIUM,8156,74,1278


In [109]:
display(
    fingpt_df[
        fingpt_df["content_quality"] == "LOW"
    ][
        [
            "id",
            "symbol",
            "company_name",
            "event_type",
            "importance",
            "content"
        ]
    ].head(50)
)

,id,symbol,company_name,event_type,importance,content
864,1406,RELIANCE,Reliance Industries Limited,Updates,MEDIUM,Media Statement
1097,1779,ASAHIINDIA,Asahi India Glass Limited,General Updates,MEDIUM,BRSR FY 2025-26
1257,1981,SHIVAMILLS,Shiva Mills Limited,Shareholders meeting,MEDIUM,Outcome of 11th AGM.
1519,2366,TCIFINANCE,TCI Finance Limited,Shareholders meeting,MEDIUM,Outcome of 52nd AGM of the Company
1647,2558,VIKASECO,Vikas EcoTech Limited,Outcome of Board Meeting,HIGH,Result outcome of board Meeting
1976,3045,BAIDFIN,Baid Finserv Limited,General Updates,MEDIUM,Annual Report 2025-26 duly submitted.
2262,3415,APTECHT,Aptech Limited,General Updates,MEDIUM,Investor Presentation
2372,3581,SUNPHARMA,Sun Pharmaceutical Industries Limited,Shareholders meeting,MEDIUM,Notice of Postal Ballot
2426,3673,ALLCARGO,Allcargo Logistics Limited,General Updates,MEDIUM,Letter Send to Members
2481,3782,ASTRAL,Astral Limited,Shareholders meeting,MEDIUM,Revised Voting Result


In [110]:
display(
    fingpt_df[
        fingpt_df["content_quality"] == "LOW"
    ][
        [
            "id",
            "symbol",
            "company_name",
            "event_type",
            "importance",
            "content"
        ]
    ].head(50)
)

,id,symbol,company_name,event_type,importance,content
864,1406,RELIANCE,Reliance Industries Limited,Updates,MEDIUM,Media Statement
1097,1779,ASAHIINDIA,Asahi India Glass Limited,General Updates,MEDIUM,BRSR FY 2025-26
1257,1981,SHIVAMILLS,Shiva Mills Limited,Shareholders meeting,MEDIUM,Outcome of 11th AGM.
1519,2366,TCIFINANCE,TCI Finance Limited,Shareholders meeting,MEDIUM,Outcome of 52nd AGM of the Company
1647,2558,VIKASECO,Vikas EcoTech Limited,Outcome of Board Meeting,HIGH,Result outcome of board Meeting
1976,3045,BAIDFIN,Baid Finserv Limited,General Updates,MEDIUM,Annual Report 2025-26 duly submitted.
2262,3415,APTECHT,Aptech Limited,General Updates,MEDIUM,Investor Presentation
2372,3581,SUNPHARMA,Sun Pharmaceutical Industries Limited,Shareholders meeting,MEDIUM,Notice of Postal Ballot
2426,3673,ALLCARGO,Allcargo Logistics Limited,General Updates,MEDIUM,Letter Send to Members
2481,3782,ASTRAL,Astral Limited,Shareholders meeting,MEDIUM,Revised Voting Result


In [111]:
display(
    fingpt_df[
        (fingpt_df["content_quality"] == "MEDIUM")
    ][
        [
            "id",
            "symbol",
            "event_type",
            "importance",
            "content_length",
            "content"
        ]
    ].head(30)
)

,id,symbol,event_type,importance,content_length,content
13,279,PERSISTENT,General Updates,MEDIUM,74,Persistent Systems Limited has informed the Exchange about General Updates
42,309,BANKBARODA,Resignation of Director/KMP/SMP,HIGH,78,Bank Of Baroda has informed the Exchange about Resignation of Director/KMP/SMP
43,310,BANKBARODA,Change in Management,HIGH,67,Bank Of Baroda has informed the Exchange about change in Management
57,324,LEAPIND,Investor Presentation,MEDIUM,72,LEAP India Limited has informed the Exchange about Investor Presentation
65,332,JMFINANCIL,Credit Rating,HIGH,66,JM Financial Limited has informed the Exchange about Credit Rating
67,334,NLCINDIA,General Updates,MEDIUM,65,NLC India Limited has informed the Exchange about General Updates
69,336,HAPPSTMNDS,Agreements,HIGH,78,Happiest Minds Technologies Limited has informed the Exchange about Agreements
70,337,LAXMIDENTL,General Updates,MEDIUM,68,Laxmi Dental Limited has informed the Exchange about General Updates
73,340,AARTISURF,Acquisition,HIGH,69,Aarti Surfactants Limited has informed the Exchange about Acquisition
77,344,AIRAN,Resignation of Director/KMP/SMP,HIGH,77,Airan Limited has informed the Exchange about Resignation of Director/KMP/SMP


In [112]:
def classify_fingpt_relevance(row):
    event = str(row["event_type"]).lower().strip()
    content = str(row["content"]).lower().strip()
    importance = str(row["importance"]).upper().strip()
    quality = str(row["content_quality"]).upper().strip()

    text = f"{event} {content}"

    # --------------------------------------------------
    # 1. Clearly material events
    # --------------------------------------------------

    high_relevance_events = [
        "acquisition",
        "amalgamation",
        "merger",
        "demerger",
        "scheme of arrangement",
        "change in management",
        "resignation",
        "cessation",
        "change in auditors",
        "credit rating",
        "bagging/receiving of orders/contracts",
        "order",
        "capacity addition",
        "commencement commercial production",
        "product launch",
        "disruption/closure operations",
        "litigation",
        "insolvency",
        "forensic audit",
        "fraud",
        "default",
        "arrest",
        "qip",
        "preferential issue",
        "rights issue",
        "buyback",
        "open offer",
        "offer for sale",
        "allotment of securities",
        "action(s) taken or orders passed",
        "disclosure of material issue",
        "corporate debt restructuring",
        "one time settlement",
        "sale or disposal",
        "diversification/disinvestment",
        "strategic/technical/manufacturing",
        "memorandum of understanding",
        "agreements",
        "trading plan under pit",
        "regulatory approvals"
    ]

    if any(keyword in event for keyword in high_relevance_events):
        return "HIGH"

    # --------------------------------------------------
    # 2. Important content keywords
    # --------------------------------------------------

    material_keywords = [
        "acquisition",
        "acquire",
        "merger",
        "demerger",
        "joint venture",
        "stake",
        "order worth",
        "contract worth",
        "crore",
        "million",
        "billion",
        "credit rating upgrade",
        "credit rating downgrade",
        "rating upgrade",
        "rating downgrade",
        "litigation",
        "penalty",
        "fine",
        "default",
        "insolvency",
        "fraud",
        "forensic",
        "regulatory approval",
        "license",
        "capacity",
        "commercial production",
        "new plant",
        "expansion",
        "dividend",
        "buyback",
        "fund raising",
        "fundraise",
        "securities",
        "appointment of chief executive",
        "appointment of chief financial officer",
        "chief financial officer",
        "managing director",
        "chief executive officer"
    ]

    if any(keyword in text for keyword in material_keywords):
        return "HIGH"

    # --------------------------------------------------
    # 3. Very weak / routine events
    # --------------------------------------------------

    low_relevance_events = [
        "trading window",
        "spurt in volume",
        "price movement",
        "loss of share certificates",
        "structural digital database",
        "monitoring agency report"
    ]

    if any(keyword in event for keyword in low_relevance_events):
        return "LOW"

    # --------------------------------------------------
    # 4. Very short generic announcements
    # --------------------------------------------------

    if quality == "LOW":
        return "LOW"

    generic_low_phrases = [
        "annual report",
        "newspaper publication",
        "letter send to members",
        "interaction with electronic media",
        "opening of branch office",
        "investor presentation"
    ]

    if any(keyword in text for keyword in generic_low_phrases):
        return "LOW"

    # --------------------------------------------------
    # 5. Shareholder / board / general updates
    # --------------------------------------------------

    if event in [
        "shareholders meeting",
        "general updates",
        "updates",
        "press release",
        "outcome of board meeting",
        "appointment"
    ]:
        if quality == "GOOD":
            return "MEDIUM"

        return "LOW"

    # --------------------------------------------------
    # 6. Existing importance as fallback
    # --------------------------------------------------

    if importance == "HIGH":
        return "HIGH"

    return "MEDIUM"

In [113]:
fingpt_df["fingpt_relevance"] = fingpt_df.apply(
    classify_fingpt_relevance,
    axis=1
)

display(
    fingpt_df["fingpt_relevance"]
    .value_counts()
    .to_frame("count")
)

,count
fingpt_relevance,
MEDIUM,8197
HIGH,3745
LOW,2109


In [114]:
display(
    fingpt_df.groupby(
        ["importance", "fingpt_relevance"]
    ).size().unstack(fill_value=0)
)

fingpt_relevance,HIGH,LOW,MEDIUM
importance,,,
HIGH,2470,52,2021
MEDIUM,1275,2057,6176


In [115]:
for relevance in ["HIGH", "MEDIUM", "LOW"]:
    print("\n" + "=" * 80)
    print(f"RELEVANCE: {relevance}")
    print("=" * 80)

    display(
        fingpt_df[
            fingpt_df["fingpt_relevance"] == relevance
        ][
            [
                "id",
                "symbol",
                "event_type",
                "importance",
                "content_quality",
                "content"
            ]
        ].sample(
            min(
                10,
                len(
                    fingpt_df[
                        fingpt_df["fingpt_relevance"] == relevance
                    ]
                )
            ),
            random_state=42
        )
    )


RELEVANCE: HIGH


,id,symbol,event_type,importance,content_quality,content
12774,19137,MAPMYINDIA,General Updates,MEDIUM,GOOD,"C.E. Info Systems Limited has informed the Exchange about confirmation of the appointment of Mr. Rakesh Kumar Verma, Group Chairman & Managing Director of the Company, as a Non-Executive Director on the Board of Mappls DT Private Limited, Material Subsidiary in the Annual General Meeting held on 3rd August, 2026."
3576,5329,BIKAJI,Appointment,MEDIUM,GOOD,"Bikaji Foods International Limited has informed the Exchange regarding Re-Appointment of Mr Deepak Agarwal as Chairman & Managing Director of the company w.e.f. February 01, 2027 at 31st Annual General Meeting."
1189,1896,SAMMAANCAP,Updates,MEDIUM,GOOD,"Sammaan Capital Limited has informed the Exchange regarding 'Pursuant to Regulation 57 of Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015, we hereby certify that our Company has made timely payment of interest amount in respect of the Secured Redeemable Non-Convertible Debentures, issued by our Company, through Public Issue, and listed on the Stock Exchanges.'."
7412,11136,JIOFIN,General Updates,MEDIUM,GOOD,"Investment of up to Rs. 18,268.22 crore by NB Holdings Corporation, wholly owned subsidiary of Bank of America Corporation in Jio Credit Limited, wholly owned subsidiary of the Company"
8392,12489,RELINFRA,General Updates,MEDIUM,GOOD,"Disclosure under Regulation 30 of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015"
7930,11788,VASCONEQ,Awarding of order(s)/contract(s),HIGH,GOOD,Vascon Engineers Limited has informed the Exchange about Awarding of order(s)/contract(s)
265,564,HITECHCORP,Record Date,MEDIUM,GOOD,Hitech Corporation Limited has informed the Exchange that Record date for the purpose of Dividend is 04-Sep-2026.
1222,1937,GUJENERGY,Updates,MEDIUM,GOOD,"GUJARAT ENERGY LIMITED has informed the Exchange regarding 'Intimation for date of payment of dividend, if declared by the Shareholders in ensuing Annual General Meeting scheduled on 29.09.2026"
7168,10742,MAHASTEEL,Disclosure of material issue,HIGH,GOOD,Mahamaya Steel Industries Limited has informed regarding Disclosure of material issue
3900,5810,KIMS,Allotment of Securities,HIGH,GOOD,Krishna Institute of Medical Sciences Limited has informed the Exchange regarding allotment of 7702182 fully convertible warrants securities pursuant to Preferential Issue



RELEVANCE: MEDIUM


,id,symbol,event_type,importance,content_quality,content
6982,10468,SPENCERS,Outcome of Board Meeting,HIGH,GOOD,"Spencer's Retail Limited has submitted to the Exchange, the financial results for the period ended Jun 30, 2026."
2295,3462,LUPIN,ESOP/ESOS/ESPS,MEDIUM,MEDIUM,Lupin Limited has informed the Exchange regarding Allotment of 30967 Shares.
10615,15778,ASHOKAMET,Change in Director(s),MEDIUM,GOOD,Ashoka Metcast Limited has informed the Exchange regarding Change in Director(s) of the company.
4264,6388,NORTHARC,Shareholders meeting,MEDIUM,GOOD,"Northern Arc Capital Limited has submitted the Exchange a copy Srutinizers report of Annual General Meeting held on August 18, 2026. Further, the company has informed the Exchange regarding voting results."
12024,17956,IFGLEXPOR,Shareholders meeting,MEDIUM,GOOD,"IFGL Refractories Limited has informed the Exchange regarding Proceedings of Annual General Meeting held on August 05, 2026"
11107,16600,PFOCUS,Appointment,MEDIUM,GOOD,"Prime Focus Limited has informed the Exchange regarding change in designation of Mr . Namit Malhotra as Whole Time Director and Key Managerial Personnel of the company w.e.f. August 07, 2026."
4400,6596,KIRLOSIND,Shareholders meeting,MEDIUM,GOOD,"Kirloskar Industries Limited has submitted the Exchange a copy Srutinizers report of Annual General Meeting held on August 18, 2026. Further, the company has informed the Exchange regarding voting results."
2703,4059,EIEL,General Updates,MEDIUM,GOOD,Enviro Infra Engineers Limited has informed the Exchange about General Updates - Letter to Shareholders who have not registered their email-ids
458,823,EXXARO,Shareholders meeting,MEDIUM,GOOD,"Exxaro Tiles Limited has informed the Exchange regarding Notice of Annual General Meeting to be held on September 26, 2026"
1254,1977,DEEDEV,Updates,MEDIUM,GOOD,"DEE Development Engineers Limited has informed the Exchange regarding Compliance Of Regulation 36(1)(B) Of The SEBI (Listing Obligations And Disclosure Requirements) Regulations, 2015'."



RELEVANCE: LOW


,id,symbol,event_type,importance,content_quality,content
3739,5557,RUBYMILLS,General Updates,MEDIUM,MEDIUM,The Ruby Mills Limited has informed the Exchange about General Updates
8485,12594,SPAL,General Updates,MEDIUM,MEDIUM,S. P. Apparels Limited has informed the Exchange about General Updates
9842,14574,MBEL,Investor Presentation,MEDIUM,MEDIUM,M & B Engineering Limited has informed the Exchange about Investor Presentation
4738,7135,SBFC,General Updates,MEDIUM,MEDIUM,Sbfc Finance Limited has informed the Exchange about General Updates
3399,5086,CYIENT,General Updates,MEDIUM,MEDIUM,Cyient Limited has informed the Exchange about General Updates
2151,3269,GPIL,Updates,MEDIUM,GOOD,Godawari Power And Ispat limited has informed the Exchange regarding 'Letter to Shareholders regarding weblink of Annual Report for FY 2025-26'.
4491,6731,CAPTRUST,General Updates,MEDIUM,GOOD,Capital Trust Limited has informed the Exchange about General Updates- Investor Presentation
12207,18235,BIOCON,Monitoring Agency Report,MEDIUM,MEDIUM,"Monitoring Agency Report for the quarter ended on June 30, 2026"
11532,17194,JKIL,General Updates,MEDIUM,MEDIUM,J.Kumar Infraprojects Limited has informed the Exchange about General Updates
5382,8217,AEPL,Investor Presentation,MEDIUM,GOOD,Artemis Electricals and Projects Limited has informed the Exchange about Investor Presentation


In [116]:
def classify_fingpt_relevance(row):
    event = str(row["event_type"]).lower().strip()
    content = str(row["content"]).lower().strip()

    importance = str(row["importance"]).upper().strip()
    quality = str(row["content_quality"]).upper().strip()

    text = f"{event} {content}"

    # --------------------------------------------------
    # 1. Clearly useless / empty announcements
    # --------------------------------------------------

    useless_phrases = [
        "general updates",
        "investor presentation",
        "monitoring agency report",
        "press release",
        "outcome of board meeting",
        "shareholders meeting",
        "updates"
    ]

    # Only classify as LOW when the announcement
    # contains almost no additional information.
    if quality == "LOW":
        return "LOW"

    if (
        quality == "MEDIUM"
        and any(phrase == event for phrase in useless_phrases)
        and len(content) < 90
    ):
        return "LOW"

    # --------------------------------------------------
    # 2. Material keywords override generic event type
    # --------------------------------------------------

    material_keywords = [
        "acquisition",
        "acquire",
        "merger",
        "demerger",
        "joint venture",
        "stake",
        "order worth",
        "contract worth",
        "crore",
        "million",
        "billion",
        "credit rating upgrade",
        "credit rating downgrade",
        "rating upgrade",
        "rating downgrade",
        "litigation",
        "penalty",
        "fine",
        "default",
        "insolvency",
        "fraud",
        "forensic",
        "regulatory approval",
        "license",
        "capacity",
        "commercial production",
        "new plant",
        "expansion",
        "dividend",
        "buyback",
        "fund raising",
        "fundraise",
        "preferential issue",
        "rights issue",
        "allotment",
        "securities",
        "chief executive officer",
        "chief financial officer",
        "managing director",
        "whole time director"
    ]

    if any(keyword in text for keyword in material_keywords):
        return "HIGH"

    # --------------------------------------------------
    # 3. Strong event types
    # --------------------------------------------------

    high_relevance_events = [
        "acquisition",
        "amalgamation",
        "merger",
        "demerger",
        "scheme of arrangement",
        "change in management",
        "resignation",
        "cessation",
        "change in auditors",
        "credit rating",
        "bagging/receiving of orders/contracts",
        "awarding of order(s)/contract(s)",
        "capacity addition",
        "commencement commercial production",
        "product launch",
        "disruption/closure operations",
        "litigation",
        "insolvency",
        "forensic audit",
        "fraud",
        "default",
        "qip",
        "preferential issue",
        "rights issue",
        "buyback",
        "open offer",
        "offer for sale",
        "allotment of securities",
        "disclosure of material issue",
        "corporate debt restructuring",
        "one time settlement",
        "sale or disposal",
        "diversification/disinvestment",
        "agreements"
    ]

    if any(keyword in event for keyword in high_relevance_events):
        return "HIGH"

    # --------------------------------------------------
    # 4. Preserve NSE HIGH importance
    # --------------------------------------------------

    if importance == "HIGH":
        return "HIGH"

    # --------------------------------------------------
    # 5. Routine MEDIUM events
    # --------------------------------------------------

    low_events = [
        "monitoring agency report",
        "investor presentation"
    ]

    if event in low_events:
        return "LOW"

    # --------------------------------------------------
    # 6. Generic shareholder / update events
    # --------------------------------------------------

    if event in [
        "shareholders meeting",
        "general updates",
        "updates",
        "press release"
    ]:
        if quality == "GOOD":
            return "MEDIUM"

        return "LOW"

    # --------------------------------------------------
    # 7. Remaining MEDIUM events
    # --------------------------------------------------

    return "MEDIUM"

In [117]:
fingpt_df["fingpt_relevance"] = fingpt_df.apply(
    classify_fingpt_relevance,
    axis=1
)

display(
    fingpt_df["fingpt_relevance"]
    .value_counts()
    .to_frame("count")
)

,count
fingpt_relevance,
MEDIUM,6270
HIGH,5985
LOW,1796


In [118]:
display(
    fingpt_df.groupby(
        ["importance", "fingpt_relevance"]
    ).size().unstack(fill_value=0)
)

fingpt_relevance,HIGH,LOW,MEDIUM
importance,,,
HIGH,4488,55,0
MEDIUM,1497,1741,6270


In [119]:
high_to_low = fingpt_df[
    (fingpt_df["importance"] == "HIGH") &
    (fingpt_df["fingpt_relevance"] == "LOW")
][
    [
        "id",
        "symbol",
        "company_name",
        "event_type",
        "importance",
        "content_quality",
        "content_length",
        "content"
    ]
].copy()

print("HIGH → LOW:", len(high_to_low))

display(high_to_low.to_string(index=False))

HIGH → LOW: 55


'   id     symbol                                       company_name               event_type importance content_quality  content_length                                                                         content\n  349    FRACTAL                          Fractal Analytics Limited Outcome of Board Meeting       HIGH          MEDIUM              53                           Appointment of Chief Financial Officer of the Company\n  432 HAPPSTMNDS                Happiest Minds Technologies Limited Outcome of Board Meeting       HIGH          MEDIUM              46                                  Outcome of Board Meeting dated August 31, 2026\n  622  MERCURYEV                            Mercury Ev-Tech Limited Outcome of Board Meeting       HIGH          MEDIUM              48                                Outcome of Board Meeting held on August 31, 2026\n  797    MANOMAY                          Manomay Tex India Limited Outcome of Board Meeting       HIGH          MEDIUM            

In [120]:
display(high_to_low.head(100))

,id,symbol,company_name,event_type,importance,content_quality,content_length,content
81,349,FRACTAL,Fractal Analytics Limited,Outcome of Board Meeting,HIGH,MEDIUM,53,Appointment of Chief Financial Officer of the Company
151,432,HAPPSTMNDS,Happiest Minds Technologies Limited,Outcome of Board Meeting,HIGH,MEDIUM,46,"Outcome of Board Meeting dated August 31, 2026"
307,622,MERCURYEV,Mercury Ev-Tech Limited,Outcome of Board Meeting,HIGH,MEDIUM,48,"Outcome of Board Meeting held on August 31, 2026"
437,797,MANOMAY,Manomay Tex India Limited,Outcome of Board Meeting,HIGH,MEDIUM,74,re-appointment of WTD and Appointment of (Additional) Independent Director
685,1140,KHAITANLTD,Khaitan (India) Limited,Outcome of Board Meeting,HIGH,MEDIUM,78,Approval of Notice of Annual General Meeting to be held on 25th September 2026
780,1284,SAKUMA,Sakuma Exports Limited,Outcome of Board Meeting,HIGH,MEDIUM,43,Outcome of Board meeting held on 29.08.2026
1647,2558,VIKASECO,Vikas EcoTech Limited,Outcome of Board Meeting,HIGH,LOW,31,Result outcome of board Meeting
2228,3369,PIGL,Power & Instrumentation (Gujarat) Limited,Outcome of Board Meeting,HIGH,MEDIUM,47,Outcome of Board Meeting held on 25 august 2026
2485,3789,TEAMLEASE,Teamlease Services Limited,Outcome of Board Meeting,HIGH,MEDIUM,60,"TeamLease - Outcome of Board Meeting held on August 25, 2026"
2971,4435,AARTISURF,Aarti Surfactants Limited,Outcome of Board Meeting,HIGH,LOW,24,Outcome of Board Meeting


In [121]:
low_records = fingpt_df[
    fingpt_df["fingpt_relevance"] == "LOW"
].copy()

material_pattern = (
    r"acquisition|acquire|merger|demerger|"
    r"joint venture|stake|order worth|contract worth|"
    r"credit rating upgrade|credit rating downgrade|"
    r"litigation|penalty|fine|default|insolvency|"
    r"fraud|forensic|regulatory approval|"
    r"capacity|commercial production|expansion|"
    r"dividend|buyback|preferential issue|rights issue|"
    r"allotment|crore|million|billion|"
    r"chief executive officer|chief financial officer|"
    r"managing director"
)

possible_false_low = low_records[
    low_records["content"]
    .str.lower()
    .str.contains(material_pattern, regex=True, na=False)
]

print("Potential LOW records containing material keywords:",
      len(possible_false_low))

display(
    possible_false_low[
        [
            "id",
            "symbol",
            "event_type",
            "importance",
            "content_quality",
            "content"
        ]
    ].head(100)
)

Potential LOW records containing material keywords: 34


,id,symbol,event_type,importance,content_quality,content
81,349,FRACTAL,Outcome of Board Meeting,HIGH,MEDIUM,Appointment of Chief Financial Officer of the Company
1352,2125,MSTCLTD,General Updates,MEDIUM,MEDIUM,Mstc Limited has informed the Exchange about imposition of fine by NSE and BSE.
1442,2263,IREDA,General Updates,MEDIUM,MEDIUM,Intimation regarding imposition of fines by Stock Exchanges.
1755,2693,OIL,General Updates,MEDIUM,MEDIUM,Oil India Limited has informed the Exchange about fine imposed by NSE and BSE
1796,2750,GRWRHITECH,General Updates,MEDIUM,MEDIUM,Communication to Shareholders - Intimation on Tax Deduction on Dividend
2009,3098,BFUTILITIE,General Updates,MEDIUM,MEDIUM,BF Utilities Limited has informed the Exchange about intimation of fine
2010,3100,RECLTD,General Updates,MEDIUM,MEDIUM,Intimation regarding imposition of fines by Stock Exchanges.
2092,3202,HONASA,General Updates,MEDIUM,MEDIUM,Update on proposed acquisition of Fluence Pharma Private Limited
2749,4114,LINC,Updates,MEDIUM,MEDIUM,Linc Limited has informed the Exchange regarding 'TDS on Dividend'.
3364,5033,RSWM,General Updates,MEDIUM,LOW,Update on Joint Venture Company


In [122]:
def classify_fingpt_relevance(row):
    event = str(row["event_type"]).lower().strip()
    content = str(row["content"]).lower().strip()

    importance = str(row["importance"]).upper().strip()
    quality = str(row["content_quality"]).upper().strip()

    text = f"{event} {content}"

    # ==================================================
    # 1. Truly empty / useless announcement
    # ==================================================

    if quality == "LOW":
        return "LOW"

    # ==================================================
    # 2. MATERIAL CONTENT — highest priority
    # ==================================================

    material_keywords = [
        "acquisition",
        "acquire",
        "merger",
        "demerger",
        "joint venture",
        "stake",
        "order worth",
        "contract worth",
        "crore",
        "million",
        "billion",

        "credit rating upgrade",
        "credit rating downgrade",
        "rating upgrade",
        "rating downgrade",

        "litigation",
        "antitrust",
        "penalty",
        "fine",
        "default",
        "insolvency",
        "fraud",
        "forensic",

        "regulatory approval",
        "regulatory order",
        "license",

        "capacity addition",
        "capacity expansion",
        "commercial production",
        "new plant",
        "expansion",

        "dividend",
        "buyback",
        "fund raising",
        "fundraise",
        "preferential issue",
        "rights issue",
        "allotment",
        "securities",

        "chief executive officer",
        "chief financial officer",
        "managing director",
        "whole time director",
        "key managerial personnel",

        "esop",
        "secondary acquisition"
    ]

    if any(keyword in text for keyword in material_keywords):
        return "HIGH"

    # ==================================================
    # 3. Strong event types
    # ==================================================

    high_relevance_events = [
        "acquisition",
        "amalgamation",
        "merger",
        "demerger",
        "scheme of arrangement",
        "change in management",
        "resignation",
        "cessation",
        "change in auditors",
        "credit rating",
        "bagging/receiving of orders/contracts",
        "awarding of order(s)/contract(s)",
        "capacity addition",
        "commencement commercial production",
        "product launch",
        "disruption/closure operations",
        "litigation",
        "insolvency",
        "forensic audit",
        "fraud",
        "default",
        "qip",
        "preferential issue",
        "rights issue",
        "buyback",
        "open offer",
        "offer for sale",
        "allotment of securities",
        "disclosure of material issue",
        "corporate debt restructuring",
        "one time settlement",
        "sale or disposal",
        "diversification/disinvestment",
        "agreements"
    ]

    if any(keyword in event for keyword in high_relevance_events):
        return "HIGH"

    # ==================================================
    # 4. Preserve NSE HIGH importance
    # ==================================================

    if importance == "HIGH":
        return "HIGH"

    # ==================================================
    # 5. Routine events
    # ==================================================

    low_events = [
        "monitoring agency report",
        "investor presentation"
    ]

    if event in low_events:
        return "LOW"

    # ==================================================
    # 6. Generic events
    # ==================================================

    if event in [
        "shareholders meeting",
        "general updates",
        "updates",
        "press release"
    ]:
        if quality == "GOOD":
            return "MEDIUM"

        return "LOW"

    # ==================================================
    # 7. Remaining records
    # ==================================================

    return "MEDIUM"

In [123]:
fingpt_df["fingpt_relevance"] = fingpt_df.apply(
    classify_fingpt_relevance,
    axis=1
)

print(
    fingpt_df["fingpt_relevance"]
    .value_counts()
)

display(
    fingpt_df.groupby(
        ["importance", "fingpt_relevance"]
    ).size().unstack(fill_value=0)
)

fingpt_relevance
MEDIUM    6214
HIGH      6109
LOW       1728
Name: count, dtype: int64


fingpt_relevance,HIGH,LOW,MEDIUM
importance,,,
HIGH,4522,21,0
MEDIUM,1587,1707,6214


In [124]:
high_to_low = fingpt_df[
    (fingpt_df["importance"] == "HIGH") &
    (fingpt_df["fingpt_relevance"] == "LOW")
][[
    "id",
    "symbol",
    "company_name",
    "event_type",
    "content",
    "importance",
    "content_quality"
]].copy()

print("HIGH → LOW:", len(high_to_low))

display(high_to_low)

HIGH → LOW: 21


,id,symbol,company_name,event_type,content,importance,content_quality
1647,2558,VIKASECO,Vikas EcoTech Limited,Outcome of Board Meeting,Result outcome of board Meeting,HIGH,LOW
2971,4435,AARTISURF,Aarti Surfactants Limited,Outcome of Board Meeting,Outcome of Board Meeting,HIGH,LOW
3337,4984,GLOBE,GLOBE ENTERPRISES (INDIA) LIMITED,Outcome of Board Meeting,Outcome of Board Meeting,HIGH,LOW
4202,6273,ABDL,Allied Blenders and Distillers Limited,Credit Rating- Revision,Credit Rating Upgrade & Assigned,HIGH,LOW
4678,7024,UGROCAP,Ugro Capital Limited,Allotment of Securities,Allotment of Commercial Paper,HIGH,LOW
5120,7922,KINGFA,Kingfa Science & Technology (India) Limited,Outcome of Board Meeting,Financial results for the Q1,HIGH,LOW
6328,9616,SBGLP,Suratwwala Business Group Limited,Outcome of Board Meeting,Outcome of board meeting,HIGH,LOW
6509,9834,JINDALPHOT,Jindal Photo Limited,Outcome of Board Meeting,"For the quarter ended June 30, 2026",HIGH,LOW
7629,11397,APCL,Anjani Portland Cement Limited,Outcome of Board Meeting,Outcome of the Board meeting,HIGH,LOW
7680,11462,CAPTRUST,Capital Trust Limited,Outcome of Board Meeting,"Outcome of the board ,meeting",HIGH,LOW


In [125]:
for _, row in high_to_low.iterrows():
    print("=" * 100)
    print("ID:", row["id"])
    print("Company:", row["company_name"], "|", row["symbol"])
    print("Event:", row["event_type"])
    print("Importance:", row["importance"])
    print("Quality:", row["content_quality"])
    print("Content:", row["content"])

ID: 2558
Company: Vikas EcoTech Limited | VIKASECO
Event: Outcome of Board Meeting
Importance: HIGH
Quality: LOW
Content: Result outcome of board Meeting
ID: 4435
Company: Aarti Surfactants Limited | AARTISURF
Event: Outcome of Board Meeting
Importance: HIGH
Quality: LOW
Content: Outcome of Board Meeting
ID: 4984
Company: GLOBE ENTERPRISES (INDIA) LIMITED | GLOBE
Event: Outcome of Board Meeting
Importance: HIGH
Quality: LOW
Content: Outcome of Board Meeting
ID: 6273
Company: Allied Blenders and Distillers Limited | ABDL
Event: Credit Rating- Revision
Importance: HIGH
Quality: LOW
Content: Credit Rating Upgrade & Assigned
ID: 7024
Company: Ugro Capital Limited | UGROCAP
Event: Allotment of Securities
Importance: HIGH
Quality: LOW
Content: Allotment of Commercial Paper
ID: 7922
Company: Kingfa Science & Technology (India) Limited | KINGFA
Event: Outcome of Board Meeting
Importance: HIGH
Quality: LOW
Content: Financial results for the Q1
ID: 9616
Company: Suratwwala Business Group Limited

In [147]:
import re
import numpy as np


# ============================================================
# TEXT NORMALIZATION
# ============================================================

def normalize_text(text):
    """
    Normalize announcement text.

    Examples:
        Credit Rating- Revision
        -> credit rating revision

        Re-Appointment of Statutory Auditors
        -> re appointment of statutory auditors
    """

    if text is None:
        return ""

    text = str(text).lower().strip()

    # Remove punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text


# ============================================================
# FINGPT RELEVANCE CLASSIFIER
# ============================================================

def classify_fingpt_relevance(row):

    event = normalize_text(row["event_type"])
    content = normalize_text(row["content"])

    importance = str(row["importance"]).upper().strip()
    quality = str(row["content_quality"]).upper().strip()

    text = f"{event} {content}"


    # ========================================================
    # 1. CLEARLY MATERIAL CONTENT
    # ========================================================
    #
    # Content itself contains a financially/corporately
    # significant event.
    #
    # This check happens BEFORE content quality.
    # Short text can still be highly relevant.
    # ========================================================

    material_keywords = [

        # ----------------------------------------------------
        # M&A / RESTRUCTURING
        # ----------------------------------------------------

        "acquisition",
        "acquire",
        "acquired",
        "merger",
        "demerger",
        "amalgamation",
        "joint venture",
        "scheme of arrangement",
        "secondary acquisition",

        "sale of",
        "sale or disposal",
        "disposal of",

        "diversification",
        "disinvestment",

        # ----------------------------------------------------
        # ORDERS / CONTRACTS
        # ----------------------------------------------------
        #
        # Avoid bare "order" / "contract".
        # Use material phrasing.
        # ----------------------------------------------------

        "order worth",
        "orders worth",
        "contract worth",
        "contracts worth",

        "bagging of order",
        "bagging of orders",
        "bagging order",
        "bagging orders",

        "receipt of order",
        "receipt of orders",
        "receipt of contract",
        "receipt of contracts",

        "received order",
        "received orders",
        "received contract",
        "received contracts",

        "awarded order",
        "awarded orders",
        "awarded contract",
        "awarded contracts",

        "landmark order",
        "major order",
        "large order",
        "significant order",

        # ----------------------------------------------------
        # CREDIT / FINANCIAL RISK
        # ----------------------------------------------------

        "credit rating upgrade",
        "credit rating downgrade",
        "credit rating revision",
        "credit rating assigned",

        "rating upgrade",
        "rating downgrade",
        "rating revision",

        "credit rating",

        "default on payment",
        "default on repayment",
        "default on interest",
        "default on principal",

        "insolvency",
        "one time settlement",
        "corporate debt restructuring",

        # ----------------------------------------------------
        # LEGAL / REGULATORY
        # ----------------------------------------------------

        "litigation",
        "antitrust",
        "tax litigation",

        "regulatory approval",
        "regulatory order",
        "regulatory action",

        "license approval",
        "licence approval",

        "penalty",
        "penalties",
        "fine",
        "fines",

        "forensic audit",
        "fraud",

        # ----------------------------------------------------
        # CAPACITY / OPERATIONS
        # ----------------------------------------------------

        "capacity addition",
        "capacity expansion",
        "capacity increase",

        "commercial production",
        "commencement commercial production",

        "new plant",
        "plant expansion",

        # ----------------------------------------------------
        # FUND RAISING / SECURITIES
        # ----------------------------------------------------

        "fund raising",
        "fundraise",

        "preferential issue",
        "preferential allotment",

        "rights issue",

        "qip",
        "qualified institutional placement",

        "allotment of securities",
        "allotment of equity shares",
        "allotment of shares",

        "issue and allotment",
        "issue of securities",

        "buyback",
        "buyback offer",

        "open offer",
        "offer for sale",

        # ----------------------------------------------------
        # MANAGEMENT / KEY PERSONNEL
        # ----------------------------------------------------

        "chief executive officer",
        "chief financial officer",

        "managing director",
        "whole time director",

        "key managerial personnel",

        "resignation of a director",
        "resignation of director",

        "resignation of chief executive officer",
        "resignation of chief financial officer",

        "cessation of director",

        "appointment of chief executive officer",
        "appointment of chief financial officer",

        "appointment of cfo",
        "appointment of ceo",

        "appointment of managing director",
        "appointment of whole time director",

        "re appointment of managing director",
        "reappointment of managing director",

        "re appointment of whole time director",
        "reappointment of whole time director",

        # ----------------------------------------------------
        # FINANCIAL RESULTS
        # ----------------------------------------------------

        "financial result",
        "financial results",

        "quarter ended",
        "quarterly results",
        "quarter result",

        "unaudited financial results",
        "audited financial results",

        # ----------------------------------------------------
        # AUDIT
        # ----------------------------------------------------

        "statutory auditor appointment",
        "appointment of statutory auditor",
        "appointment of statutory auditors",

        "re appointment of statutory auditor",
        "re appointment of statutory auditors",

        "reappointment of statutory auditor",
        "reappointment of statutory auditors",

        # ----------------------------------------------------
        # ESOP
        # ----------------------------------------------------

        "esop",
        "employee stock option",

        "allotment pursuant to exercise of stock options",
        "allotment pursuant to exercise of esop",

        # ----------------------------------------------------
        # MATERIAL BUSINESS MILESTONE
        # ----------------------------------------------------

        "business milestone",
        "strategic milestone",
        "landmark achievement"
    ]


    if any(keyword in text for keyword in material_keywords):
        return "HIGH"


    # ========================================================
    # 2. MATERIAL EVENT TYPES
    # ========================================================

    high_relevance_events = [

        "acquisition",
        "amalgamation",
        "merger",
        "demerger",
        "scheme of arrangement",

        "change in management",
        "resignation",
        "cessation",
        "change in auditors",

        "credit rating",

        "capacity addition",
        "commencement commercial production",

        "litigation",
        "insolvency",
        "forensic audit",
        "fraud",
        "default",

        "qip",
        "preferential issue",
        "rights issue",

        "buyback",
        "open offer",
        "offer for sale",

        "allotment of securities",

        "corporate debt restructuring",
        "one time settlement",

        "sale or disposal",
        "diversification disinvestment"
    ]


    if any(keyword in event for keyword in high_relevance_events):
        return "HIGH"


    # ========================================================
    # 3. ORIGINAL HIGH + MEANINGFUL CONTENT
    # ========================================================

    if importance == "HIGH" and quality != "LOW":
        return "HIGH"
    # ========================================================
    # 4. MONITORING AGENCY REPORT
    # ========================================================
    #
    # Generic monitoring reports are not useful enough by
    # themselves.
    #
    # However, if the content mentions rights issue,
    # preferential issue, IPO, or utilisation of funds,
    # the material-keyword section above has already returned
    # HIGH.
    # ========================================================
    
    if event == "monitoring agency report":
        return "LOW"
    
    
    # ========================================================
    # 5. INVESTOR PRESENTATION
    # ========================================================
    #
    # The NSE feed generally gives only the fact that an
    # investor presentation was submitted.
    #
    # Unless the actual material content is available,
    # don't send it to FinGPT.
    # ========================================================
    
    if event == "investor presentation":
        return "LOW"
    
    
    # ========================================================
    # 6. SHAREHOLDERS MEETING
    # ========================================================
    #
    # Generic AGM notices / proceedings / voting results
    # are not HIGH by themselves.
    #
    # Material events such as merger, scheme, buyback etc.
    # have already been caught above.
    # ========================================================
    
    if event == "shareholders meeting":
    
        if quality == "LOW":
            return "LOW"
    
        return "MEDIUM"


    # ========================================================
    # 7. GENERIC BOARD-MEETING CONTENT
    # ========================================================

    generic_low_content = [

        "outcome of board meeting",
        "result outcome of board meeting",
        "outcome of the board meeting",
        "outcome of board meeting",
        "financial result"
    ]


    if content in generic_low_content:
        return "LOW"


    # ========================================================
    # 8. GENERIC MEDIUM EVENTS
    # ========================================================

    generic_medium_events = [

        "shareholders meeting",
        "general updates",
        "updates",
        "press release",
        "press release revised",
        "press release revised"
    ]


    if event in generic_medium_events:

        if quality == "GOOD":
            return "MEDIUM"

        return "LOW"


    # ========================================================
    # 9. LOW-VALUE EVENTS
    # ========================================================

    low_events = [

        "monitoring agency report",
        "investor presentation"
    ]


    if event in low_events:
        return "LOW"


    # ========================================================
    # 10. DEFAULT
    # ========================================================

    return "MEDIUM"

In [148]:
fingpt_df["fingpt_relevance"] = fingpt_df.apply(
    classify_fingpt_relevance,
    axis=1
)

print(fingpt_df["fingpt_relevance"].value_counts())

display(
    pd.crosstab(
        fingpt_df["importance"],
        fingpt_df["fingpt_relevance"]
    )
)

fingpt_relevance
MEDIUM    6562
HIGH      6067
LOW       1422
Name: count, dtype: int64


fingpt_relevance,HIGH,LOW,MEDIUM
importance,,,
HIGH,4530,13,0
MEDIUM,1537,1409,6562


In [149]:
# HIGH -> LOW
high_to_low = fingpt_df[
    (fingpt_df["importance"] == "HIGH") &
    (fingpt_df["fingpt_relevance"] == "LOW")
][[
    "id",
    "symbol",
    "company_name",
    "event_type",
    "content",
    "importance",
    "content_quality"
]]

print("HIGH → LOW:", len(high_to_low))

display(high_to_low)

HIGH → LOW: 13


,id,symbol,company_name,event_type,content,importance,content_quality
1647,2558,VIKASECO,Vikas EcoTech Limited,Outcome of Board Meeting,Result outcome of board Meeting,HIGH,LOW
2971,4435,AARTISURF,Aarti Surfactants Limited,Outcome of Board Meeting,Outcome of Board Meeting,HIGH,LOW
3337,4984,GLOBE,GLOBE ENTERPRISES (INDIA) LIMITED,Outcome of Board Meeting,Outcome of Board Meeting,HIGH,LOW
6328,9616,SBGLP,Suratwwala Business Group Limited,Outcome of Board Meeting,Outcome of board meeting,HIGH,LOW
7629,11397,APCL,Anjani Portland Cement Limited,Outcome of Board Meeting,Outcome of the Board meeting,HIGH,LOW
7680,11462,CAPTRUST,Capital Trust Limited,Outcome of Board Meeting,"Outcome of the board ,meeting",HIGH,LOW
8993,13272,GTECJAINX,G-TEC JAINX EDUCATION LIMITED,Outcome of Board Meeting,Outcome of board meeting,HIGH,LOW
9617,14182,MVGJL,Manoj Vaibhav Gems N Jewellers Limited,Outcome of Board Meeting,Outcome of Board Meeting,HIGH,LOW
10013,14879,POWERMECH,Power Mech Projects Limited,Outcome of Board Meeting,Outcome of Board meeting,HIGH,LOW
10550,15673,PFC,Power Finance Corporation Limited,Outcome of Board Meeting,Outcome of Board Meeting,HIGH,LOW


In [150]:
medium_to_high = fingpt_df[
    (fingpt_df["importance"] == "MEDIUM") &
    (fingpt_df["fingpt_relevance"] == "HIGH")
][[
    "id",
    "symbol",
    "company_name",
    "event_type",
    "content",
    "importance",
    "content_quality"
]]

print("MEDIUM → HIGH:", len(medium_to_high))

display(medium_to_high.head(100))

MEDIUM → HIGH: 1537


,id,symbol,company_name,event_type,content,importance,content_quality
7,273,WELSPUNLIV,Welspun Living Limited,General Updates,Welspun Living Limited has informed the Exchange about Update on acquisition of 26% equity stake in Clean Max Dhyuthi Private Limited.,MEDIUM,GOOD
49,316,VERANDA,Veranda Learning Solutions Limited,General Updates,"Intimation regarding receipt of Certified True Copy of the Order of the National Company Law Tribunal, Chennai Bench and implementation of the Composite Scheme of Arrangement",MEDIUM,GOOD
53,320,TBZ,Tribhovandas Bhimji Zaveri Limited,Updates,Tribhovandas Bhimji Zaveri Limited has informed the Exchange regarding 'Receipt of Public Announcement in relation to Open Offer'.,MEDIUM,GOOD
60,327,LEAPIND,LEAP India Limited,Press Release,"LEAP India Limited has informed the Exchange regarding a press release dated August 31, 2026, titled ""Press release for the Unaudited Financial Results for the quarter ended June 30, 2026"".",MEDIUM,GOOD
80,347,KCPSUGIND,KCP Sugar and Industries Corporation Limited,Updates,Submission of Financial results for the period ended 30th June 2026 in Machine Readable Form,MEDIUM,GOOD
...,...,...,...,...,...,...,...
1490,2327,MARKOLINES,Markolines Pavement Technologies Limited,Appointment,"Markolines Pavement Technologies Limited has informed the Exchange regarding Re-appointment of Mr. Sanjay Bhanudas Patil as Managing Director of the company w.e.f. August 27, 2026.",MEDIUM,GOOD
1497,2335,SDBL,Som Distilleries & Breweries Limited,Updates,"Som Distilleries & Breweries Limited has informed the Exchange regarding 'Outcome of Board Meeting held on Tuesday 11th August, 2026 - financial results for the period ended 30-Jun-2026 in Machine Readable Form / Legible copy.'.",MEDIUM,GOOD
1520,2367,ANIKINDS,Anik Industries Limited,Updates,Anik Industries Limited has informed the Exchange regarding ''Board of Directors Comments on fine levied by the Stock Exchange'.'.,MEDIUM,GOOD
1542,2416,WAKEFIT,Wakefit Innovations Limited,ESOP/ESOS/ESPS,"The Board of Directors vide resolution passed on August 27, 2026, has approved the allotment of 3,13,521 Equity shares of Re.1 /- each to eligible employees/ ESOP holders.",MEDIUM,GOOD


In [151]:
# Final FinGPT dataset
fingpt_final_df = fingpt_df[
    fingpt_df["fingpt_relevance"].isin(["HIGH", "MEDIUM"])
].copy()

print("Total FinGPT records:", len(fingpt_final_df))
print(fingpt_final_df["fingpt_relevance"].value_counts())

Total FinGPT records: 12629
fingpt_relevance
MEDIUM    6562
HIGH      6067
Name: count, dtype: int64


In [152]:
fingpt_high_df = fingpt_final_df[
    fingpt_final_df["fingpt_relevance"] == "HIGH"
].copy()

print("HIGH records:", len(fingpt_high_df))

HIGH records: 6067


In [153]:
fingpt_high_df[
    [
        "id",
        "symbol",
        "company_name",
        "event_type",
        "content",
        "importance",
        "fingpt_relevance"
    ]
].head(20)

,id,symbol,company_name,event_type,content,importance,fingpt_relevance
7,273,WELSPUNLIV,Welspun Living Limited,General Updates,Welspun Living Limited has informed the Exchange about Update on acquisition of 26% equity stake in Clean Max Dhyuthi Private Limited.,MEDIUM,HIGH
8,274,NIITMTS,NIIT Learning Systems Limited,Other Restructuring,"Update of the MST Holding GmbH Germany, a step down wholly owned subsidiary (""WoS"") of the Company for the voluntary dissolution and winding up of its Wos - MST Shanghai Co. Limited.",HIGH,HIGH
10,276,BRITANNIA,Britannia Industries Limited,Change in Management,"Pursuant to Regulation 30 read with Clause 7 of Para A of Part A of Schedule III of the SEBI Listing Regulations, 2015, this is to inform you that Mr. Susheel Navanale, Chief Information Officer of the Company will retire from the services of the Company with effect from close of business hours on 30th September, 2026. In this regard, the Board of Directors, based on the recommendation of the Nomination and Remuneration Committee, vide their Circular Resolution passed today i.e. 31st Augus...",HIGH,HIGH
12,278,TEXRAIL,Texmaco Rail & Engineering Limited,Resignation of Director/KMP/SMP,Texmaco Rail & Engineering Limited has informed the Exchange about Resignation of Director/KMP/SMP,HIGH,HIGH
15,281,LICHSGFIN,LIC Housing Finance Limited,Change in Management,LIC Housing Finance Limited has informed the Exchange about change in Senior Management-Cessation of I Sreedhar,HIGH,HIGH
16,282,SWELECTES,Swelect Energy Systems Limited,Sale or disposal,SWELECT Energy Systems Limited has informed the Exchange about Transfer of 26% equity shares held in ESG Solar Energy Private Limited,HIGH,HIGH
17,283,TEXRAIL,Texmaco Rail & Engineering Limited,Resignation,"Texmaco Rail & Engineering Limited has informed the Exchange regarding Resignation of Mr. Kishor Kumar Rajgaria as Chief Financial Officer of the company w.e.f. August 31, 2026.",HIGH,HIGH
19,285,TATASTEEL,Tata Steel Limited,Diversification/Disinvestment,"Tata Steel Limited has informed the Exchange about divestment of entire stake held in Jamshedpur Football and Sporting Private Limited, wholly owned subsidiary",HIGH,HIGH
22,288,TBZ,Tribhovandas Bhimji Zaveri Limited,Public Announcement-Open Offer,"Axis Capital Limited (Manager to the Offer) has submitted to the Exchange a copy of Public Announcement for the attention of the Public Shareholders of Tribhovandas Bhimji Zaveri Limited under Regulations 3(1), 3(2) and 4 read with Regulations 13(1), 13(2), 13(2A), 14 and 15(1) of the Securities and Exchange Board of India (Substantial Acquisition of Shares and Takeovers) Regulations, 2011 and subsequent amendments thereto.",HIGH,HIGH
25,292,SIGNATURE,Signatureglobal (India) Limited,Resignation of Director/KMP/SMP,Signatureglobal (India) Limited has informed the Exchange about Resignation of Director/KMP/SMP,HIGH,HIGH


In [154]:
def build_fingpt_prompt(row):
    return f"""
You are a financial news analysis model.

Analyze the following NSE corporate announcement.

Company: {row['company_name']}
Symbol: {row['symbol']}
Event Type: {row['event_type']}
Date: {row['published_at']}

Announcement:
{row['content']}

Return:
1. Sentiment: POSITIVE / NEGATIVE / NEUTRAL
2. Event: What happened?
3. Financial Impact: HIGH / MEDIUM / LOW
4. Business Impact: HIGH / MEDIUM / LOW
5. Short-term Market Impact: POSITIVE / NEGATIVE / NEUTRAL
6. Reason: Brief explanation
""".strip()

fingpt_high_df["fingpt_prompt"] = fingpt_high_df.apply(
    build_fingpt_prompt,
    axis=1
)

In [155]:
print(fingpt_high_df.iloc[0]["fingpt_prompt"])

You are a financial news analysis model.

Analyze the following NSE corporate announcement.

Company: Welspun Living Limited
Symbol: WELSPUNLIV
Event Type: General Updates
Date: 2026-08-31 23:21:49

Announcement:
Welspun Living Limited has informed the Exchange about Update on acquisition of 26% equity stake in Clean Max Dhyuthi Private Limited.

Return:
1. Sentiment: POSITIVE / NEGATIVE / NEUTRAL
2. Event: What happened?
3. Financial Impact: HIGH / MEDIUM / LOW
4. Business Impact: HIGH / MEDIUM / LOW
5. Short-term Market Impact: POSITIVE / NEGATIVE / NEUTRAL
6. Reason: Brief explanation


In [156]:
import os

print("Models folder:")
print(os.listdir("models"))

Models folder:
[]


In [157]:
for root, dirs, files in os.walk("models"):
    level = root.replace("models", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files[:10]:
        print(f"{indent}  {file}")

models/


In [158]:
import importlib.util

packages = [
    "torch",
    "transformers",
    "accelerate",
    "bitsandbytes",
    "sentencepiece"
]

for package in packages:
    print(
        package,
        "✅" if importlib.util.find_spec(package) else "❌"
    )

torch ✅
transformers ✅
accelerate ❌
bitsandbytes ❌
sentencepiece ❌


In [159]:
import torch

print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

PyTorch: 2.13.0
MPS available: True


In [160]:
!pip install -U accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [accelerate] 1/2 [accelerate]

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [161]:
import torch
import transformers
import accelerate
import sentencepiece

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("SentencePiece: OK")
print("MPS available:", torch.backends.mps.is_available())

PyTorch: 2.13.0
Transformers: 5.15.0
Accelerate: 1.15.0
SentencePiece: OK
MPS available: True


In [163]:
import peft

print("PEFT:", peft.__version__)


PEFT: 0.21.0


In [164]:
from huggingface_hub import HfApi

api = HfApi()

model_id = "FinGPT/fingpt-mt_llama2-7b_lora"

info = api.model_info(model_id)

print("Model:", info.id)
print("Downloads:", info.downloads)
print("Files:")
for file in info.siblings:
    print(" -", file.rfilename)

Model: FinGPT/fingpt-mt_llama2-7b_lora
Downloads: 0
Files:
 - .gitattributes
 - README.md
 - adapter_config.json
 - adapter_model.bin


In [166]:
from huggingface_hub import HfApi

api = HfApi()

models = api.list_models(
    author="FinGPT",
    sort="downloads",
    limit=20
)

for model in models:
    print(
        f"{model.id:<50} "
        f"downloads={model.downloads}"
    )

FinGPT/fingpt-forecaster_dow30_llama2-7b_lora      downloads=417
FinGPT/fingpt-mt_qwen-7b_lora                      downloads=230
FinGPT/fingpt-mt_mpt-7b_lora                       downloads=13
FinGPT/fingpt-mt_llama2-7b_lora                    downloads=0
FinGPT/fingpt-mt_bloom-7b1_lora                    downloads=0
FinGPT/fingpt-mt_falcon-7b_lora                    downloads=0
FinGPT/fingpt-mt_chatglm2-6b_lora                  downloads=0
FinGPT/fingpt-sentiment_llama2-13b_lora            downloads=0
FinGPT/fingpt-sentiment_internlm-20b_lora          downloads=0
FinGPT/fingpt-forecaster_sz50_llama2-7B_lora       downloads=0
FinGPT/fingpt-mt_llama3-8b_lora                    downloads=0


In [167]:
import psutil

ram_gb = psutil.virtual_memory().total / (1024**3)

print(f"Total RAM: {ram_gb:.2f} GB")
print(f"Available RAM: {psutil.virtual_memory().available / (1024**3):.2f} GB")

Total RAM: 8.00 GB
Available RAM: 1.46 GB


In [168]:
import platform

print("Machine:", platform.machine())
print("Processor:", platform.processor())

Machine: arm64
Processor: arm
